In [1]:
## sort NQ data

import pandas as pd
from collections import defaultdict

# Load the data
file_path = r"C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv"
df = pd.read_csv(file_path)

# Combine Date and Time into a datetime column
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df.set_index('Datetime', inplace=True)

# Filter data from 2013 to 2025
start_date = pd.to_datetime('2013-01-01')
end_date = pd.to_datetime('2025-12-31')
df = df.loc[start_date:end_date]

# Adjust the index to group by NY trading day (18:00 to 17:00)
df['NY_Trading_Day'] = (df.index - pd.Timedelta(hours=7)).normalize()
dates = df['NY_Trading_Day'].unique()

df.head(5)

,Date,Time,Open,High,Low,Close,Volume,NY_Trading_Day
Datetime,,,,,,,,
2013-01-02 06:00:00,2013-01-02,06:00:00,2686.25,2692.50,2686.25,2691.75,1555,2013-01-01
2013-01-02 06:01:00,2013-01-02,06:01:00,2692.00,2694.50,2692.00,2694.25,465,2013-01-01
2013-01-02 06:02:00,2013-01-02,06:02:00,2694.50,2697.25,2694.00,2697.00,461,2013-01-01
2013-01-02 06:03:00,2013-01-02,06:03:00,2697.00,2698.50,2696.00,2698.00,359,2013-01-01
2013-01-02 06:04:00,2013-01-02,06:04:00,2697.75,2698.00,2696.00,2697.00,189,2013-01-01


In [2]:
## Modified NQ Analysis: Session patterns, midpoint analysis, and hit rate statistics
## ADDED: Sequential hit analysis - statistics for levels hit AFTER the first level


# ============================================================
# SECTION 1: SESSION PATTERN STATISTICS (KEPT FROM ORIGINAL)
# ============================================================

# Initialize dictionaries to store stats for each pattern
stats = {
    'london_engulfs_asia': {
        'total': 0,
        'ny_high_takes_london_high': 0,
        'ny_low_takes_london_low': 0,
        'ny_engulfs_overnight': 0,
        'hod_hours': defaultdict(int),
        'lod_hours': defaultdict(int),
        'ny_cash_bullish': 0,
        'ny_cash_bearish': 0
    },
    'asia_engulfs_london': {
        'total': 0,
        'ny_high_takes_asia_high': 0,
        'ny_low_takes_asia_low': 0,
        'ny_engulfs_overnight': 0,
        'hod_hours': defaultdict(int),
        'lod_hours': defaultdict(int),
        'ny_cash_bullish': 0,
        'ny_cash_bearish': 0
    },
    'london_partially_engulfs_upwards': {
        'total': 0,
        'ny_high_takes_london_high': 0,
        'ny_low_takes_london_low': 0,
        'ny_low_takes_asia_low': 0,
        'ny_engulfs_overnight': 0,
        'hod_hours': defaultdict(int),
        'lod_hours': defaultdict(int),
        'ny_cash_bullish': 0,
        'ny_cash_bearish': 0
    },
    'london_partially_engulfs_downwards': {
        'total': 0,
        'ny_low_takes_london_low': 0,
        'ny_high_takes_london_high': 0,
        'ny_high_takes_asia_high': 0,
        'ny_engulfs_overnight': 0,
        'hod_hours': defaultdict(int),
        'lod_hours': defaultdict(int),
        'ny_cash_bullish': 0,
        'ny_cash_bearish': 0
    }
}

# Track overall HOD/LOD hours
overall_hod_hours = defaultdict(int)
overall_lod_hours = defaultdict(int)
valid_days_count = 0

# ============================================================
# SECTION 2: NEW - SESSION MIDPOINT ANALYSIS
# ============================================================

# Track session open relative to prior session midpoint
# Structure: midpoint_stats[transition_type][position] = {'hit_high_first': count, 'hit_low_first': count}
midpoint_stats = {
    'london_vs_asia': {
        'above_midpoint': {'hit_high_first': 0, 'hit_low_first': 0, 'hit_neither': 0},
        'below_midpoint': {'hit_high_first': 0, 'hit_low_first': 0, 'hit_neither': 0}
    },
    'ny_vs_london': {
        'above_midpoint': {'hit_high_first': 0, 'hit_low_first': 0, 'hit_neither': 0},
        'below_midpoint': {'hit_high_first': 0, 'hit_low_first': 0, 'hit_neither': 0}
    },
    'ny_vs_overnight': {  # NY vs combined Asia+London
        'above_midpoint': {'hit_high_first': 0, 'hit_low_first': 0, 'hit_neither': 0},
        'below_midpoint': {'hit_high_first': 0, 'hit_low_first': 0, 'hit_neither': 0}
    }
}

# ============================================================
# SECTION 3: NEW - HIT RATE STATISTICS (8 AM - 4 PM)
# ============================================================

# Track hit rates during NY session (8 AM - 4 PM)
hit_rate_stats = {
    'prev_day_high': {'hit': 0, 'total': 0},
    'prev_day_low': {'hit': 0, 'total': 0},
    'seven_to_eight_midpoint': {'hit': 0, 'total': 0},
    'midnight_open': {'hit': 0, 'total': 0}
}

# Conditional hit rates based on NY open position relative to London midpoint
conditional_hit_rates = {
    'ny_above_london_mid': {
        'seven_to_eight_midpoint': {'hit': 0, 'total': 0},
        'midnight_open': {'hit': 0, 'total': 0},
        'two_to_three_midpoint': {'hit': 0, 'total': 0}
    },
    'ny_below_london_mid': {
        'seven_to_eight_midpoint': {'hit': 0, 'total': 0},
        'midnight_open': {'hit': 0, 'total': 0},
        'two_to_three_midpoint': {'hit': 0, 'total': 0}
    }
}

# Hit rates by session pattern
pattern_hit_rates = {
    'london_engulfs_asia': {
        'ny_above_london_mid': {
            'seven_to_eight_midpoint': {'hit': 0, 'total': 0},
            'midnight_open': {'hit': 0, 'total': 0},
            'two_to_three_midpoint': {'hit': 0, 'total': 0}
        },
        'ny_below_london_mid': {
            'seven_to_eight_midpoint': {'hit': 0, 'total': 0},
            'midnight_open': {'hit': 0, 'total': 0},
            'two_to_three_midpoint': {'hit': 0, 'total': 0}
        }
    },
    'asia_engulfs_london': {
        'ny_above_london_mid': {
            'seven_to_eight_midpoint': {'hit': 0, 'total': 0},
            'midnight_open': {'hit': 0, 'total': 0},
            'two_to_three_midpoint': {'hit': 0, 'total': 0}
        },
        'ny_below_london_mid': {
            'seven_to_eight_midpoint': {'hit': 0, 'total': 0},
            'midnight_open': {'hit': 0, 'total': 0},
            'two_to_three_midpoint': {'hit': 0, 'total': 0}
        }
    },
    'london_partially_engulfs_upwards': {
        'ny_above_london_mid': {
            'seven_to_eight_midpoint': {'hit': 0, 'total': 0},
            'midnight_open': {'hit': 0, 'total': 0},
            'two_to_three_midpoint': {'hit': 0, 'total': 0}
        },
        'ny_below_london_mid': {
            'seven_to_eight_midpoint': {'hit': 0, 'total': 0},
            'midnight_open': {'hit': 0, 'total': 0},
            'two_to_three_midpoint': {'hit': 0, 'total': 0}
        }
    },
    'london_partially_engulfs_downwards': {
        'ny_above_london_mid': {
            'seven_to_eight_midpoint': {'hit': 0, 'total': 0},
            'midnight_open': {'hit': 0, 'total': 0},
            'two_to_three_midpoint': {'hit': 0, 'total': 0}
        },
        'ny_below_london_mid': {
            'seven_to_eight_midpoint': {'hit': 0, 'total': 0},
            'midnight_open': {'hit': 0, 'total': 0},
            'two_to_three_midpoint': {'hit': 0, 'total': 0}
        }
    }
}

# ============================================================
# SECTION 4: NEW - DISTANCE FROM 8 AM OPEN TO KEY LEVELS
# ============================================================

# Track % distance from 8am open to each level
distance_from_8am_open = {
    'seven_to_eight_midpoint': [],
    'midnight_open': [],
    'two_to_three_midpoint': [],
    'prev_day_high': [],
    'prev_day_low': []
}

# ============================================================
# SECTION 5: NEW - RANGE EXTENSION ANALYSIS
# ============================================================

# Track how far sessions extend in terms of prior session range units
range_extensions = {
    'london_above_asia': [],      # Units above Asia high
    'london_below_asia': [],      # Units below Asia low
    'ny_above_asia': [],          # Units above Asia high
    'ny_below_asia': [],          # Units below Asia low
    'ny_above_london': [],        # Units above London high
    'ny_below_london': [],        # Units below London low
    'ny_above_overnight': [],     # Units above overnight high
    'ny_below_overnight': []      # Units below overnight low
}

# ============================================================
# SECTION 6: NEW - SEQUENTIAL HIT ANALYSIS FOR SESSION LEVELS
# ============================================================

# Track hits WITHIN each session, CONDITIONAL on where session opens relative to prior session midpoint

# LONDON SESSION: Track Asia high/low hits conditional on London open vs Asia midpoint
london_sequential_hits = {
    'above_midpoint': {  # London opens above Asia midpoint
        'asia_high': {
            'asia_low': {'hit': 0, 'total': 0}
        },
        'asia_low': {
            'asia_high': {'hit': 0, 'total': 0}
        }
    },
    'below_midpoint': {  # London opens below Asia midpoint
        'asia_high': {
            'asia_low': {'hit': 0, 'total': 0}
        },
        'asia_low': {
            'asia_high': {'hit': 0, 'total': 0}
        }
    }
}

london_first_hit_count = {
    'above_midpoint': {
        'asia_high': 0,
        'asia_low': 0,
        'none': 0
    },
    'below_midpoint': {
        'asia_high': 0,
        'asia_low': 0,
        'none': 0
    }
}

# NY SESSION: Track all prior session levels conditional on NY open vs Overnight midpoint
ny_sequential_hits = {
    'above_midpoint': {  # NY opens above Overnight midpoint
        'asia_high': {
            'asia_low': {'hit': 0, 'total': 0},
            'london_high': {'hit': 0, 'total': 0},
            'london_low': {'hit': 0, 'total': 0},
            'overnight_high': {'hit': 0, 'total': 0},
            'overnight_low': {'hit': 0, 'total': 0}
        },
        'asia_low': {
            'asia_high': {'hit': 0, 'total': 0},
            'london_high': {'hit': 0, 'total': 0},
            'london_low': {'hit': 0, 'total': 0},
            'overnight_high': {'hit': 0, 'total': 0},
            'overnight_low': {'hit': 0, 'total': 0}
        },
        'london_high': {
            'asia_high': {'hit': 0, 'total': 0},
            'asia_low': {'hit': 0, 'total': 0},
            'london_low': {'hit': 0, 'total': 0},
            'overnight_high': {'hit': 0, 'total': 0},
            'overnight_low': {'hit': 0, 'total': 0}
        },
        'london_low': {
            'asia_high': {'hit': 0, 'total': 0},
            'asia_low': {'hit': 0, 'total': 0},
            'london_high': {'hit': 0, 'total': 0},
            'overnight_high': {'hit': 0, 'total': 0},
            'overnight_low': {'hit': 0, 'total': 0}
        },
        'overnight_high': {
            'asia_high': {'hit': 0, 'total': 0},
            'asia_low': {'hit': 0, 'total': 0},
            'london_high': {'hit': 0, 'total': 0},
            'london_low': {'hit': 0, 'total': 0},
            'overnight_low': {'hit': 0, 'total': 0}
        },
        'overnight_low': {
            'asia_high': {'hit': 0, 'total': 0},
            'asia_low': {'hit': 0, 'total': 0},
            'london_high': {'hit': 0, 'total': 0},
            'london_low': {'hit': 0, 'total': 0},
            'overnight_high': {'hit': 0, 'total': 0}
        }
    },
    'below_midpoint': {  # NY opens below Overnight midpoint
        'asia_high': {
            'asia_low': {'hit': 0, 'total': 0},
            'london_high': {'hit': 0, 'total': 0},
            'london_low': {'hit': 0, 'total': 0},
            'overnight_high': {'hit': 0, 'total': 0},
            'overnight_low': {'hit': 0, 'total': 0}
        },
        'asia_low': {
            'asia_high': {'hit': 0, 'total': 0},
            'london_high': {'hit': 0, 'total': 0},
            'london_low': {'hit': 0, 'total': 0},
            'overnight_high': {'hit': 0, 'total': 0},
            'overnight_low': {'hit': 0, 'total': 0}
        },
        'london_high': {
            'asia_high': {'hit': 0, 'total': 0},
            'asia_low': {'hit': 0, 'total': 0},
            'london_low': {'hit': 0, 'total': 0},
            'overnight_high': {'hit': 0, 'total': 0},
            'overnight_low': {'hit': 0, 'total': 0}
        },
        'london_low': {
            'asia_high': {'hit': 0, 'total': 0},
            'asia_low': {'hit': 0, 'total': 0},
            'london_high': {'hit': 0, 'total': 0},
            'overnight_high': {'hit': 0, 'total': 0},
            'overnight_low': {'hit': 0, 'total': 0}
        },
        'overnight_high': {
            'asia_high': {'hit': 0, 'total': 0},
            'asia_low': {'hit': 0, 'total': 0},
            'london_high': {'hit': 0, 'total': 0},
            'london_low': {'hit': 0, 'total': 0},
            'overnight_low': {'hit': 0, 'total': 0}
        },
        'overnight_low': {
            'asia_high': {'hit': 0, 'total': 0},
            'asia_low': {'hit': 0, 'total': 0},
            'london_high': {'hit': 0, 'total': 0},
            'london_low': {'hit': 0, 'total': 0},
            'overnight_high': {'hit': 0, 'total': 0}
        }
    }
}

ny_first_hit_count = {
    'above_midpoint': {
        'asia_high': 0,
        'asia_low': 0,
        'london_high': 0,
        'london_low': 0,
        'overnight_high': 0,
        'overnight_low': 0,
        'none': 0
    },
    'below_midpoint': {
        'asia_high': 0,
        'asia_low': 0,
        'london_high': 0,
        'london_low': 0,
        'overnight_high': 0,
        'overnight_low': 0,
        'none': 0
    }
}

# ============================================================
# MAIN ANALYSIS LOOP
# ============================================================

for date in dates[1:]:  # Start from the second day
    # Map the NY trading day back to the actual date
    actual_date = date + pd.Timedelta(hours=7)

    # Define the full NY trading day range
    ny_trading_day_start = actual_date.replace(hour=18, minute=0, second=0) - pd.Timedelta(days=1)
    ny_trading_day_end = actual_date.replace(hour=17, minute=0, second=0)

    # Get all data for this NY trading day
    full_day_data = df.loc[ny_trading_day_start:ny_trading_day_end]
    
    if full_day_data.empty:
        continue

    # Find HOD and LOD for the entire trading day
    day_high = full_day_data['High'].max()
    day_low = full_day_data['Low'].min()
    
    # Find the hour when HOD occurred
    hod_time = full_day_data[full_day_data['High'] == day_high].index[0]
    hod_hour = hod_time.hour
    
    # Find the hour when LOD occurred
    lod_time = full_day_data[full_day_data['Low'] == day_low].index[0]
    lod_hour = lod_time.hour
    
    # Track overall HOD/LOD hours
    overall_hod_hours[hod_hour] += 1
    overall_lod_hours[lod_hour] += 1
    valid_days_count += 1

    # Define session time ranges
    asian_start = actual_date.replace(hour=20, minute=0, second=0) - pd.Timedelta(days=1)
    asian_end = actual_date.replace(hour=2, minute=0, second=0)
    london_start = asian_end
    london_end = actual_date.replace(hour=8, minute=0, second=0)
    ny_start = london_end
    ny_end = actual_date.replace(hour=16, minute=0, second=0)

    # Extract session data
    asian_data = df.loc[asian_start:asian_end]
    london_data = df.loc[london_start:london_end]
    ny_data = df.loc[ny_start:ny_end]

    # Check for early closure in NY session
    if not ny_data.empty:
        last_timestamp = ny_data.index.max()
        if last_timestamp < ny_end - pd.Timedelta(minutes=30):
            ny_end = last_timestamp
            ny_data = df.loc[ny_start:ny_end]

    # Skip if any session data is missing
    if asian_data.empty or london_data.empty or ny_data.empty:
        continue

    # Calculate high and low for each session
    asian_high = asian_data['High'].max()
    asian_low = asian_data['Low'].min()
    london_high = london_data['High'].max()
    london_low = london_data['Low'].min()
    ny_high = ny_data['High'].max()
    ny_low = ny_data['Low'].min()

    # Skip if any high/low is NaN
    if pd.isna([asian_high, asian_low, london_high, london_low, ny_high, ny_low]).any():
        continue

    # Calculate overnight session range
    overnight_high = max(asian_high, london_high)
    overnight_low = min(asian_low, london_low)

    # ============================================================
    # SEQUENTIAL HIT ANALYSIS - LONDON SESSION
    # ============================================================
    
    # Determine London open position relative to Asia midpoint
    asia_midpoint = (asian_high + asian_low) / 2
    london_open = london_data['Open'].iloc[0] if not london_data.empty else None
    
    if london_open is not None:
        if london_open > asia_midpoint:
            london_position = 'above_midpoint'
        else:
            london_position = 'below_midpoint'
        
        # Track which Asia level is hit first during London session
        asia_levels_london = {
            'asia_high': asian_high,
            'asia_low': asian_low
        }
        
        asia_hit_times_london = {}
        
        # Track when each Asia level is hit during London session
        for idx, row in london_data.iterrows():
            for level_name, level_price in asia_levels_london.items():
                if level_name not in asia_hit_times_london:
                    if row['Low'] <= level_price <= row['High']:
                        asia_hit_times_london[level_name] = idx
        
        # Determine which Asia level was hit first in London session
        if asia_hit_times_london:
            first_hit_london = min(asia_hit_times_london, key=asia_hit_times_london.get)
            first_hit_time_london = asia_hit_times_london[first_hit_london]
            london_first_hit_count[london_position][first_hit_london] += 1
            
            # Check if OTHER Asia level gets hit after the first hit
            subsequent_london_data = london_data.loc[first_hit_time_london:]
            
            for level_name, level_price in asia_levels_london.items():
                if level_name != first_hit_london:
                    london_sequential_hits[london_position][first_hit_london][level_name]['total'] += 1
                    
                    # Check if this level was hit in subsequent London data
                    for idx, row in subsequent_london_data.iterrows():
                        if row['Low'] <= level_price <= row['High']:
                            london_sequential_hits[london_position][first_hit_london][level_name]['hit'] += 1
                            break
        else:
            london_first_hit_count[london_position]['none'] += 1

    # ============================================================
    # SESSION PATTERN CLASSIFICATION (ORIGINAL CODE)
    # ============================================================
    
    current_pattern = None

    # Pattern 1: London Engulfs Asia
    if london_high > asian_high and london_low < asian_low:
        current_pattern = 'london_engulfs_asia'
        stats['london_engulfs_asia']['total'] += 1
        if ny_high > london_high:
            stats['london_engulfs_asia']['ny_high_takes_london_high'] += 1
        if ny_low < london_low:
            stats['london_engulfs_asia']['ny_low_takes_london_low'] += 1
        if ny_high > overnight_high and ny_low < overnight_low:
            stats['london_engulfs_asia']['ny_engulfs_overnight'] += 1

    # Pattern 2: Asia Engulfs London
    elif london_high <= asian_high and london_low >= asian_low:
        current_pattern = 'asia_engulfs_london'
        stats['asia_engulfs_london']['total'] += 1
        if ny_high > asian_high:
            stats['asia_engulfs_london']['ny_high_takes_asia_high'] += 1
        if ny_low < asian_low:
            stats['asia_engulfs_london']['ny_low_takes_asia_low'] += 1
        if ny_high > overnight_high and ny_low < overnight_low:
            stats['asia_engulfs_london']['ny_engulfs_overnight'] += 1

    # Pattern 3: London Partially Engulfs Upwards
    elif london_high > asian_high and london_low >= asian_low:
        current_pattern = 'london_partially_engulfs_upwards'
        stats['london_partially_engulfs_upwards']['total'] += 1
        if ny_high > london_high:
            stats['london_partially_engulfs_upwards']['ny_high_takes_london_high'] += 1
        if ny_low < london_low:
            stats['london_partially_engulfs_upwards']['ny_low_takes_london_low'] += 1
        if ny_low < asian_low:
            stats['london_partially_engulfs_upwards']['ny_low_takes_asia_low'] += 1
        if ny_high > overnight_high and ny_low < overnight_low:
            stats['london_partially_engulfs_upwards']['ny_engulfs_overnight'] += 1

    # Pattern 4: London Partially Engulfs Downwards
    elif london_low < asian_low and london_high <= asian_high:
        current_pattern = 'london_partially_engulfs_downwards'
        stats['london_partially_engulfs_downwards']['total'] += 1
        if ny_low < london_low:
            stats['london_partially_engulfs_downwards']['ny_low_takes_london_low'] += 1
        if ny_high > london_high:
            stats['london_partially_engulfs_downwards']['ny_high_takes_london_high'] += 1
        if ny_high > asian_high:
            stats['london_partially_engulfs_downwards']['ny_high_takes_asia_high'] += 1
        if ny_high > overnight_high and ny_low < overnight_low:
            stats['london_partially_engulfs_downwards']['ny_engulfs_overnight'] += 1

    # Track HOD/LOD hours for this pattern
    if current_pattern:
        stats[current_pattern]['hod_hours'][hod_hour] += 1
        stats[current_pattern]['lod_hours'][lod_hour] += 1
        
        # Track NY cash session direction
        ny_cash_start = actual_date.replace(hour=9, minute=30, second=0)
        ny_cash_end = actual_date.replace(hour=16, minute=0, second=0)
        
        ny_cash_data = df.loc[ny_cash_start:ny_cash_end]
        
        if not ny_cash_data.empty:
            ny_cash_open = ny_cash_data['Open'].iloc[0]
            ny_cash_close = ny_cash_data['Close'].iloc[-1]
            
            if ny_cash_close > ny_cash_open:
                stats[current_pattern]['ny_cash_bullish'] += 1
            elif ny_cash_close < ny_cash_open:
                stats[current_pattern]['ny_cash_bearish'] += 1

    # ============================================================
    # NEW: SESSION MIDPOINT ANALYSIS
    # ============================================================
    
    # 1. London open vs Asia midpoint
    asia_midpoint = (asian_high + asian_low) / 2
    london_open = london_data['Open'].iloc[0] if not london_data.empty else None
    
    if london_open is not None:
        if london_open > asia_midpoint:
            position = 'above_midpoint'
        else:
            position = 'below_midpoint'
        
        # Check which is hit first during London session: Asia high or Asia low
        hit_high_first = False
        hit_low_first = False
        
        for idx, row in london_data.iterrows():
            if not hit_high_first and not hit_low_first:
                if row['High'] >= asian_high:
                    hit_high_first = True
                    midpoint_stats['london_vs_asia'][position]['hit_high_first'] += 1
                    break
                elif row['Low'] <= asian_low:
                    hit_low_first = True
                    midpoint_stats['london_vs_asia'][position]['hit_low_first'] += 1
                    break
        
        if not hit_high_first and not hit_low_first:
            midpoint_stats['london_vs_asia'][position]['hit_neither'] += 1
    
    # 2. NY open vs London midpoint
    london_midpoint = (london_high + london_low) / 2
    ny_open = ny_data['Open'].iloc[0] if not ny_data.empty else None
    
    if ny_open is not None:
        if ny_open > london_midpoint:
            position = 'above_midpoint'
        else:
            position = 'below_midpoint'
        
        # Check which is hit first during NY session: London high or London low
        hit_high_first = False
        hit_low_first = False
        
        for idx, row in ny_data.iterrows():
            if not hit_high_first and not hit_low_first:
                if row['High'] >= london_high:
                    hit_high_first = True
                    midpoint_stats['ny_vs_london'][position]['hit_high_first'] += 1
                    break
                elif row['Low'] <= london_low:
                    hit_low_first = True
                    midpoint_stats['ny_vs_london'][position]['hit_low_first'] += 1
                    break
        
        if not hit_high_first and not hit_low_first:
            midpoint_stats['ny_vs_london'][position]['hit_neither'] += 1
    
    # 3. NY open vs Overnight (Asia + London) midpoint
    overnight_midpoint = (overnight_high + overnight_low) / 2
    
    if ny_open is not None:
        if ny_open > overnight_midpoint:
            position = 'above_midpoint'
        else:
            position = 'below_midpoint'
        
        # Check which is hit first during NY session: Overnight high or Overnight low
        hit_high_first = False
        hit_low_first = False
        
        for idx, row in ny_data.iterrows():
            if not hit_high_first and not hit_low_first:
                if row['High'] >= overnight_high:
                    hit_high_first = True
                    midpoint_stats['ny_vs_overnight'][position]['hit_high_first'] += 1
                    break
                elif row['Low'] <= overnight_low:
                    hit_low_first = True
                    midpoint_stats['ny_vs_overnight'][position]['hit_low_first'] += 1
                    break
        
        if not hit_high_first and not hit_low_first:
            midpoint_stats['ny_vs_overnight'][position]['hit_neither'] += 1

    # ============================================================
    # NEW: RANGE EXTENSION ANALYSIS
    # ============================================================
    
    # Calculate range extensions as % of the reference price level
    # This normalizes across different price levels over the 12-year period
    
    # London extensions relative to Asia (as % of Asia high/low)
    if asian_high > 0:
        if london_high > asian_high:
            london_above_asia_pct = ((london_high - asian_high) / asian_high) * 100
            range_extensions['london_above_asia'].append(london_above_asia_pct)
        else:
            range_extensions['london_above_asia'].append(0)
    
    if asian_low > 0:
        if london_low < asian_low:
            london_below_asia_pct = ((asian_low - london_low) / asian_low) * 100
            range_extensions['london_below_asia'].append(london_below_asia_pct)
        else:
            range_extensions['london_below_asia'].append(0)
    
    # NY extensions relative to Asia (as % of Asia high/low)
    if asian_high > 0:
        if ny_high > asian_high:
            ny_above_asia_pct = ((ny_high - asian_high) / asian_high) * 100
            range_extensions['ny_above_asia'].append(ny_above_asia_pct)
        else:
            range_extensions['ny_above_asia'].append(0)
    
    if asian_low > 0:
        if ny_low < asian_low:
            ny_below_asia_pct = ((asian_low - ny_low) / asian_low) * 100
            range_extensions['ny_below_asia'].append(ny_below_asia_pct)
        else:
            range_extensions['ny_below_asia'].append(0)
    
    # NY extensions relative to London (as % of London high/low)
    if london_high > 0:
        if ny_high > london_high:
            ny_above_london_pct = ((ny_high - london_high) / london_high) * 100
            range_extensions['ny_above_london'].append(ny_above_london_pct)
        else:
            range_extensions['ny_above_london'].append(0)
    
    if london_low > 0:
        if ny_low < london_low:
            ny_below_london_pct = ((london_low - ny_low) / london_low) * 100
            range_extensions['ny_below_london'].append(ny_below_london_pct)
        else:
            range_extensions['ny_below_london'].append(0)
    
    # NY extensions relative to Overnight (as % of Overnight high/low)
    if overnight_high > 0:
        if ny_high > overnight_high:
            ny_above_overnight_pct = ((ny_high - overnight_high) / overnight_high) * 100
            range_extensions['ny_above_overnight'].append(ny_above_overnight_pct)
        else:
            range_extensions['ny_above_overnight'].append(0)
    
    if overnight_low > 0:
        if ny_low < overnight_low:
            ny_below_overnight_pct = ((overnight_low - ny_low) / overnight_low) * 100
            range_extensions['ny_below_overnight'].append(ny_below_overnight_pct)
        else:
            range_extensions['ny_below_overnight'].append(0)

    # ============================================================
    # NEW: HIT RATE ANALYSIS (8 AM - 4 PM)
    # ============================================================
    
    # Define the analysis window (8 AM - 4 PM)
    analysis_start = actual_date.replace(hour=8, minute=0, second=0)
    analysis_end = actual_date.replace(hour=16, minute=0, second=0)
    analysis_data = df.loc[analysis_start:analysis_end]
    
    if not analysis_data.empty:
        # Get 8 AM open price for distance calculations
        eight_am_open = analysis_data['Open'].iloc[0]
        analysis_high = analysis_data['High'].max()
        analysis_low = analysis_data['Low'].min()
        
        # Determine NY open position relative to London midpoint
        london_midpoint = (london_high + london_low) / 2
        ny_open = ny_data['Open'].iloc[0] if not ny_data.empty else None
        
        if ny_open is not None:
            if ny_open > london_midpoint:
                ny_position = 'ny_above_london_mid'
            else:
                ny_position = 'ny_below_london_mid'
        else:
            ny_position = None
        
        # ============================================================
        # CALCULATE INTRADAY PRICE LEVELS (for original hit rate stats)
        # ============================================================
        
        # 1. Previous day high/low
        prev_date = date - pd.Timedelta(days=1)
        prev_day_actual_date = prev_date + pd.Timedelta(hours=7)
        prev_day_start = prev_day_actual_date.replace(hour=18, minute=0, second=0) - pd.Timedelta(days=1)
        prev_day_end = prev_day_actual_date.replace(hour=17, minute=0, second=0)
        
        prev_day_data = df.loc[prev_day_start:prev_day_end]
        
        if not prev_day_data.empty:
            prev_day_high = prev_day_data['High'].max()
            prev_day_low = prev_day_data['Low'].min()
        else:
            prev_day_high = None
            prev_day_low = None
        
        # 2. 7-8 AM hourly midpoint
        seven_to_eight_start = actual_date.replace(hour=7, minute=0, second=0)
        seven_to_eight_end = actual_date.replace(hour=8, minute=0, second=0)
        seven_to_eight_data = df.loc[seven_to_eight_start:seven_to_eight_end]
        
        if not seven_to_eight_data.empty:
            seven_to_eight_high = seven_to_eight_data['High'].max()
            seven_to_eight_low = seven_to_eight_data['Low'].min()
            seven_to_eight_midpoint = (seven_to_eight_high + seven_to_eight_low) / 2
        else:
            seven_to_eight_midpoint = None
        
        # 3. Midnight open
        midnight_time = actual_date.replace(hour=0, minute=0, second=0)
        midnight_data = df.loc[midnight_time:]
        
        if not midnight_data.empty:
            midnight_open = midnight_data['Open'].iloc[0]
        else:
            midnight_open = None
        
        # 4. 2-3 AM hourly midpoint
        two_to_three_start = actual_date.replace(hour=2, minute=0, second=0)
        two_to_three_end = actual_date.replace(hour=3, minute=0, second=0)
        two_to_three_data = df.loc[two_to_three_start:two_to_three_end]
        
        if not two_to_three_data.empty:
            two_to_three_high = two_to_three_data['High'].max()
            two_to_three_low = two_to_three_data['Low'].min()
            two_to_three_midpoint = (two_to_three_high + two_to_three_low) / 2
        else:
            two_to_three_midpoint = None
        
        # ============================================================
        # SEQUENTIAL HIT ANALYSIS - NY SESSION
        # ============================================================
        
        # Determine NY open position relative to Overnight midpoint
        overnight_midpoint = (overnight_high + overnight_low) / 2
        ny_open = ny_data['Open'].iloc[0] if not ny_data.empty else None
        
        if ny_open is not None:
            if ny_open > overnight_midpoint:
                ny_position_midpoint = 'above_midpoint'
            else:
                ny_position_midpoint = 'below_midpoint'
            
            # Define all prior session levels available to NY session
            ny_session_levels = {
                'asia_high': asian_high,
                'asia_low': asian_low,
                'london_high': london_high,
                'london_low': london_low,
                'overnight_high': overnight_high,
                'overnight_low': overnight_low
            }
            
            # Dictionary to track when each session level is hit during NY session
            session_level_hit_times = {}
            
            # Iterate through NY session (8 AM - 4 PM) and track when each session level is hit
            for idx, row in analysis_data.iterrows():
                for level_name, level_price in ny_session_levels.items():
                    # If level not yet hit and price touched this bar, record the time
                    if level_name not in session_level_hit_times:
                        if row['Low'] <= level_price <= row['High']:
                            session_level_hit_times[level_name] = idx
            
            # Determine which session level was hit first
            if session_level_hit_times:
                first_hit_level = min(session_level_hit_times, key=session_level_hit_times.get)
                first_hit_time = session_level_hit_times[first_hit_level]
                ny_first_hit_count[ny_position_midpoint][first_hit_level] += 1
                
                # Now check which OTHER session levels got hit AFTER the first hit
                subsequent_data = analysis_data.loc[first_hit_time:]
                
                for level_name, level_price in ny_session_levels.items():
                    if level_name != first_hit_level:  # Don't count the first hit level
                        ny_sequential_hits[ny_position_midpoint][first_hit_level][level_name]['total'] += 1
                        
                        # Check if this level was hit in the subsequent data
                        for idx, row in subsequent_data.iterrows():
                            if row['Low'] <= level_price <= row['High']:
                                ny_sequential_hits[ny_position_midpoint][first_hit_level][level_name]['hit'] += 1
                                break
            else:
                ny_first_hit_count[ny_position_midpoint]['none'] += 1
        
        # ============================================================
        # ORIGINAL HIT RATE TRACKING (UNCHANGED)
        # ============================================================
        
        if not prev_day_data.empty:
            hit_rate_stats['prev_day_high']['total'] += 1
            hit_rate_stats['prev_day_low']['total'] += 1
            
            if analysis_high >= prev_day_high:
                hit_rate_stats['prev_day_high']['hit'] += 1
            if analysis_low <= prev_day_low:
                hit_rate_stats['prev_day_low']['hit'] += 1
            
            # Track % distance from 8am open to prev day levels
            if eight_am_open > 0:
                pct_dist_to_high = ((prev_day_high - eight_am_open) / eight_am_open) * 100
                pct_dist_to_low = ((prev_day_low - eight_am_open) / eight_am_open) * 100
                distance_from_8am_open['prev_day_high'].append(pct_dist_to_high)
                distance_from_8am_open['prev_day_low'].append(pct_dist_to_low)
        
        if not seven_to_eight_data.empty:
            hit_rate_stats['seven_to_eight_midpoint']['total'] += 1
            
            # Check if price touched the midpoint during analysis window
            hit_seven_eight = analysis_low <= seven_to_eight_midpoint <= analysis_high
            if hit_seven_eight:
                hit_rate_stats['seven_to_eight_midpoint']['hit'] += 1
            
            # Track % distance from 8am open to 7-8am midpoint
            if eight_am_open > 0:
                pct_dist_to_midpoint = ((seven_to_eight_midpoint - eight_am_open) / eight_am_open) * 100
                distance_from_8am_open['seven_to_eight_midpoint'].append(pct_dist_to_midpoint)
            
            # Conditional tracking based on NY position
            if ny_position is not None:
                conditional_hit_rates[ny_position]['seven_to_eight_midpoint']['total'] += 1
                if hit_seven_eight:
                    conditional_hit_rates[ny_position]['seven_to_eight_midpoint']['hit'] += 1
                
                # Pattern-specific tracking
                if current_pattern is not None:
                    pattern_hit_rates[current_pattern][ny_position]['seven_to_eight_midpoint']['total'] += 1
                    if hit_seven_eight:
                        pattern_hit_rates[current_pattern][ny_position]['seven_to_eight_midpoint']['hit'] += 1
        
        if not midnight_data.empty:
            hit_rate_stats['midnight_open']['total'] += 1
            
            # Check if price touched the midnight open during analysis window
            hit_midnight = analysis_low <= midnight_open <= analysis_high
            if hit_midnight:
                hit_rate_stats['midnight_open']['hit'] += 1
            
            # Track % distance from 8am open to midnight open
            if eight_am_open > 0:
                pct_dist_to_midnight = ((midnight_open - eight_am_open) / eight_am_open) * 100
                distance_from_8am_open['midnight_open'].append(pct_dist_to_midnight)
            
            # Conditional tracking based on NY position
            if ny_position is not None:
                conditional_hit_rates[ny_position]['midnight_open']['total'] += 1
                if hit_midnight:
                    conditional_hit_rates[ny_position]['midnight_open']['hit'] += 1
                
                # Pattern-specific tracking
                if current_pattern is not None:
                    pattern_hit_rates[current_pattern][ny_position]['midnight_open']['total'] += 1
                    if hit_midnight:
                        pattern_hit_rates[current_pattern][ny_position]['midnight_open']['hit'] += 1
        
        if not two_to_three_data.empty:
            # Track % distance from 8am open to 2-3am midpoint
            if eight_am_open > 0:
                pct_dist_to_two_three = ((two_to_three_midpoint - eight_am_open) / eight_am_open) * 100
                distance_from_8am_open['two_to_three_midpoint'].append(pct_dist_to_two_three)
            
            # Conditional tracking based on NY position
            if ny_position is not None:
                hit_two_three = analysis_low <= two_to_three_midpoint <= analysis_high
                
                conditional_hit_rates[ny_position]['two_to_three_midpoint']['total'] += 1
                if hit_two_three:
                    conditional_hit_rates[ny_position]['two_to_three_midpoint']['hit'] += 1
                
                # Pattern-specific tracking
                if current_pattern is not None:
                    pattern_hit_rates[current_pattern][ny_position]['two_to_three_midpoint']['total'] += 1
                    if hit_two_three:
                        pattern_hit_rates[current_pattern][ny_position]['two_to_three_midpoint']['hit'] += 1

# ============================================================
# CALCULATE PERCENTAGES
# ============================================================

# Pattern stats percentages (original)
for pattern_key in stats.keys():
    if stats[pattern_key]['total'] > 0:
        pattern = stats[pattern_key]
        total = pattern['total']
        
        for key in list(pattern.keys()):
            if key.endswith('_pct'):
                continue
            if key in ['total', 'hod_hours', 'lod_hours', 'ny_cash_bullish', 'ny_cash_bearish']:
                continue
            
            pct_key = key + '_pct'
            stats[pattern_key][pct_key] = round(pattern[key] / total * 100, 2)

# ============================================================
# PRINT RESULTS
# ============================================================

def create_centered_header(text, total_width, dash_char):
    text_width = len(text)
    if text_width >= total_width - 4:
        total_width = text_width + 8
    side_dashes = (total_width - text_width - 2) // 2
    return f"{dash_char * side_dashes} {text} {dash_char * side_dashes}"

desc_width = 35
header_width = 45
dash = "-"

print("\n" + "=" * 80)
print("FUTURES ANALYSIS REPORT")
print("=" * 80)
print(f"\nData Period: {start_date.date()} to {df.index.max().date()}")
print(f"Total trading days analyzed: {len(dates) - 1}")

# ============================================================
# SECTION 1: SESSION PATTERN STATISTICS (ORIGINAL)
# ============================================================

print("\n\n" + "=" * 80)
print("SECTION 1: SESSION PATTERN STATISTICS")
print("=" * 80)

print(f"\nDays with London Engulfs Asia               : {stats['london_engulfs_asia']['total']}")
print(f"Days with Asia Engulfs London               : {stats['asia_engulfs_london']['total']}")
print(f"Days with London Partially Engulfs Upwards  : {stats['london_partially_engulfs_upwards']['total']}")
print(f"Days with London Partially Engulfs Downwards: {stats['london_partially_engulfs_downwards']['total']}")

print("\n" + create_centered_header("London Engulfs Asia", header_width, dash))
if stats['london_engulfs_asia']['total'] > 0:
    print(f"{'New York high takes London high'.ljust(desc_width)} | {stats['london_engulfs_asia']['ny_high_takes_london_high_pct']:.2f}%")
    print(f"{'New York low takes London low'.ljust(desc_width)} | {stats['london_engulfs_asia']['ny_low_takes_london_low_pct']:.2f}%")
    print(f"{'New York engulfs overnight session'.ljust(desc_width)} | {stats['london_engulfs_asia']['ny_engulfs_overnight_pct']:.2f}%")
    ny_cash_total = stats['london_engulfs_asia']['ny_cash_bullish'] + stats['london_engulfs_asia']['ny_cash_bearish']
    if ny_cash_total > 0:
        bullish_pct = stats['london_engulfs_asia']['ny_cash_bullish'] / ny_cash_total * 100
        bearish_pct = stats['london_engulfs_asia']['ny_cash_bearish'] / ny_cash_total * 100
        print(f"{'NY Cash Session (9:30-4:00) Bullish'.ljust(desc_width)} | {bullish_pct:.2f}%")
        print(f"{'NY Cash Session (9:30-4:00) Bearish'.ljust(desc_width)} | {bearish_pct:.2f}%")

print("\n" + create_centered_header("Asia Engulfs London", header_width, dash))
if stats['asia_engulfs_london']['total'] > 0:
    print(f"{'New York high takes Asia high'.ljust(desc_width)} | {stats['asia_engulfs_london']['ny_high_takes_asia_high_pct']:.2f}%")
    print(f"{'New York low takes Asia low'.ljust(desc_width)} | {stats['asia_engulfs_london']['ny_low_takes_asia_low_pct']:.2f}%")
    print(f"{'New York engulfs overnight session'.ljust(desc_width)} | {stats['asia_engulfs_london']['ny_engulfs_overnight_pct']:.2f}%")
    ny_cash_total = stats['asia_engulfs_london']['ny_cash_bullish'] + stats['asia_engulfs_london']['ny_cash_bearish']
    if ny_cash_total > 0:
        bullish_pct = stats['asia_engulfs_london']['ny_cash_bullish'] / ny_cash_total * 100
        bearish_pct = stats['asia_engulfs_london']['ny_cash_bearish'] / ny_cash_total * 100
        print(f"{'NY Cash Session (9:30-4:00) Bullish'.ljust(desc_width)} | {bullish_pct:.2f}%")
        print(f"{'NY Cash Session (9:30-4:00) Bearish'.ljust(desc_width)} | {bearish_pct:.2f}%")

print("\n" + create_centered_header("London Partially Engulfs Upwards", header_width, dash))
if stats['london_partially_engulfs_upwards']['total'] > 0:
    print(f"{'New York high takes London high'.ljust(desc_width)} | {stats['london_partially_engulfs_upwards']['ny_high_takes_london_high_pct']:.2f}%")
    print(f"{'New York low takes London low'.ljust(desc_width)} | {stats['london_partially_engulfs_upwards']['ny_low_takes_london_low_pct']:.2f}%")
    print(f"{'New York low takes Asian low'.ljust(desc_width)} | {stats['london_partially_engulfs_upwards']['ny_low_takes_asia_low_pct']:.2f}%")
    print(f"{'New York engulfs overnight session'.ljust(desc_width)} | {stats['london_partially_engulfs_upwards']['ny_engulfs_overnight_pct']:.2f}%")
    ny_cash_total = stats['london_partially_engulfs_upwards']['ny_cash_bullish'] + stats['london_partially_engulfs_upwards']['ny_cash_bearish']
    if ny_cash_total > 0:
        bullish_pct = stats['london_partially_engulfs_upwards']['ny_cash_bullish'] / ny_cash_total * 100
        bearish_pct = stats['london_partially_engulfs_upwards']['ny_cash_bearish'] / ny_cash_total * 100
        print(f"{'NY Cash Session (9:30-4:00) Bullish'.ljust(desc_width)} | {bullish_pct:.2f}%")
        print(f"{'NY Cash Session (9:30-4:00) Bearish'.ljust(desc_width)} | {bearish_pct:.2f}%")

print("\n" + create_centered_header("London Partially Engulfs Downwards", header_width, dash))
if stats['london_partially_engulfs_downwards']['total'] > 0:
    print(f"{'New York low takes London low'.ljust(desc_width)} | {stats['london_partially_engulfs_downwards']['ny_low_takes_london_low_pct']:.2f}%")
    print(f"{'New York high takes London high'.ljust(desc_width)} | {stats['london_partially_engulfs_downwards']['ny_high_takes_london_high_pct']:.2f}%")
    print(f"{'New York high takes Asian high'.ljust(desc_width)} | {stats['london_partially_engulfs_downwards']['ny_high_takes_asia_high_pct']:.2f}%")
    print(f"{'New York engulfs overnight session'.ljust(desc_width)} | {stats['london_partially_engulfs_downwards']['ny_engulfs_overnight_pct']:.2f}%")
    ny_cash_total = stats['london_partially_engulfs_downwards']['ny_cash_bullish'] + stats['london_partially_engulfs_downwards']['ny_cash_bearish']
    if ny_cash_total > 0:
        bullish_pct = stats['london_partially_engulfs_downwards']['ny_cash_bullish'] / ny_cash_total * 100
        bearish_pct = stats['london_partially_engulfs_downwards']['ny_cash_bearish'] / ny_cash_total * 100
        print(f"{'NY Cash Session (9:30-4:00) Bullish'.ljust(desc_width)} | {bullish_pct:.2f}%")
        print(f"{'NY Cash Session (9:30-4:00) Bearish'.ljust(desc_width)} | {bearish_pct:.2f}%")

# ============================================================
# SECTION 2: SESSION MIDPOINT ANALYSIS
# ============================================================

print("\n\n" + "=" * 80)
print("SECTION 2: SESSION MIDPOINT ANALYSIS")
print("=" * 80)
print("\nWhen a session opens relative to the previous session's midpoint,")
print("which extreme (high or low) of the previous session is hit first?")

for transition, title in [
    ('london_vs_asia', 'London Open vs Asia Session Midpoint'),
    ('ny_vs_london', 'NY Open vs London Session Midpoint'),
    ('ny_vs_overnight', 'NY Open vs Overnight (Asia+London) Midpoint')
]:
    print("\n" + "-" * 80)
    print(f"{title}")
    print("-" * 80)
    
    for position, position_label in [
        ('above_midpoint', 'Session opens ABOVE prior session midpoint'),
        ('below_midpoint', 'Session opens BELOW prior session midpoint')
    ]:
        data = midpoint_stats[transition][position]
        total = data['hit_high_first'] + data['hit_low_first'] + data['hit_neither']
        
        if total > 0:
            high_first_pct = (data['hit_high_first'] / total) * 100
            low_first_pct = (data['hit_low_first'] / total) * 100
            neither_pct = (data['hit_neither'] / total) * 100
            
            print(f"\n  {position_label}:")
            print(f"    Total occurrences    : {total}")
            print(f"    Hit HIGH first       : {data['hit_high_first']:4d} ({high_first_pct:5.2f}%)")
            print(f"    Hit LOW first        : {data['hit_low_first']:4d} ({low_first_pct:5.2f}%)")
            print(f"    Hit neither          : {data['hit_neither']:4d} ({neither_pct:5.2f}%)")
        else:
            print(f"\n  {position_label}:")
            print(f"    No data available")

# ============================================================
# SECTION 3: HIT RATE STATISTICS (8 AM - 4 PM)
# ============================================================

print("\n\n" + "=" * 80)
print("SECTION 3: PRICE LEVEL HIT RATES (8 AM - 4 PM)")
print("=" * 80)
print("\nHow often do specific price levels get touched during the 8 AM - 4 PM session?")

print("\n" + "-" * 80)
print("OVERALL HIT RATES (unconditional)")
print("-" * 80)

for level_key, level_name in [
    ('prev_day_high', 'Previous Day High'),
    ('prev_day_low', 'Previous Day Low'),
    ('seven_to_eight_midpoint', '7-8 AM Hour Midpoint'),
    ('midnight_open', 'Midnight (00:00) Open Price')
]:
    data = hit_rate_stats[level_key]
    
    if data['total'] > 0:
        hit_rate = (data['hit'] / data['total']) * 100
        print(f"\n{level_name}:")
        print(f"  Total days analyzed : {data['total']}")
        print(f"  Days hit            : {data['hit']}")
        print(f"  Hit rate            : {hit_rate:.2f}%")
    else:
        print(f"\n{level_name}:")
        print(f"  No data available")

# ============================================================
# CONDITIONAL HIT RATES BASED ON NY OPEN POSITION
# ============================================================

print("\n\n" + "-" * 80)
print("CONDITIONAL HIT RATES BASED ON NY OPEN vs LONDON MIDPOINT")
print("-" * 80)
print("\nHit rates depend on where NY session opens relative to London session midpoint")

for position, position_label in [
    ('ny_above_london_mid', 'NY Opens ABOVE London Midpoint'),
    ('ny_below_london_mid', 'NY Opens BELOW London Midpoint')
]:
    print(f"\n\n{position_label}:")
    print("=" * 60)
    
    for level_key, level_name in [
        ('seven_to_eight_midpoint', '7-8 AM Hour Midpoint'),
        ('midnight_open', 'Midnight (00:00) Open Price'),
        ('two_to_three_midpoint', '2-3 AM Hour Midpoint')
    ]:
        data = conditional_hit_rates[position][level_key]
        
        if data['total'] > 0:
            hit_rate = (data['hit'] / data['total']) * 100
            print(f"\n  {level_name}:")
            print(f"    Total occurrences : {data['total']}")
            print(f"    Times hit         : {data['hit']}")
            print(f"    Hit rate          : {hit_rate:.2f}%")
        else:
            print(f"\n  {level_name}:")
            print(f"    No data available")

# ============================================================
# PATTERN-SPECIFIC CONDITIONAL HIT RATES
# ============================================================

print("\n\n" + "=" * 80)
print("SECTION 4: PATTERN-SPECIFIC CONDITIONAL HIT RATES")
print("=" * 80)
print("\nHit rates by session pattern AND NY open position relative to London midpoint")

pattern_display_names = {
    'london_engulfs_asia': 'London Engulfs Asia',
    'asia_engulfs_london': 'Asia Engulfs London',
    'london_partially_engulfs_upwards': 'London Partially Engulfs Upwards',
    'london_partially_engulfs_downwards': 'London Partially Engulfs Downwards'
}

for pattern_key, pattern_name in pattern_display_names.items():
    if stats[pattern_key]['total'] > 0:
        print("\n\n" + "=" * 80)
        print(f"PATTERN: {pattern_name}")
        print("=" * 80)
        
        for position, position_label in [
            ('ny_above_london_mid', 'NY Opens ABOVE London Midpoint'),
            ('ny_below_london_mid', 'NY Opens BELOW London Midpoint')
        ]:
            print(f"\n{position_label}:")
            print("-" * 60)
            
            for level_key, level_name in [
                ('seven_to_eight_midpoint', '7-8 AM Hour Midpoint'),
                ('midnight_open', 'Midnight (00:00) Open Price'),
                ('two_to_three_midpoint', '2-3 AM Hour Midpoint')
            ]:
                data = pattern_hit_rates[pattern_key][position][level_key]
                
                if data['total'] > 0:
                    hit_rate = (data['hit'] / data['total']) * 100
                    print(f"\n  {level_name}:")
                    print(f"    Total occurrences : {data['total']}")
                    print(f"    Times hit         : {data['hit']}")
                    print(f"    Hit rate          : {hit_rate:.2f}%")
                else:
                    print(f"\n  {level_name}:")
                    print(f"    No data available")

# ============================================================
# SECTION 5: DISTANCE FROM 8 AM OPEN TO KEY LEVELS
# ============================================================

print("\n\n" + "=" * 80)
print("SECTION 5: MEDIAN % DISTANCE FROM 8 AM OPEN TO KEY LEVELS")
print("=" * 80)
print("\nHow far away (in % terms) are key levels from where NY session opens at 8 AM?")
print("Positive % = level is above 8am open | Negative % = level is below 8am open")

print("\n" + "-" * 80)

for level_key, level_name in [
    ('seven_to_eight_midpoint', '7-8 AM Hour Midpoint'),
    ('midnight_open', 'Midnight (00:00) Open Price'),
    ('two_to_three_midpoint', '2-3 AM Hour Midpoint'),
    ('prev_day_high', 'Previous Day High'),
    ('prev_day_low', 'Previous Day Low')
]:
    distances = distance_from_8am_open[level_key]
    
    if distances:
        median_distance = pd.Series(distances).median()
        mean_distance = pd.Series(distances).mean()
        
        print(f"\n{level_name}:")
        print(f"  Sample size      : {len(distances)}")
        print(f"  Median distance  : {median_distance:+.3f}%")
        print(f"  Mean distance    : {mean_distance:+.3f}%")
    else:
        print(f"\n{level_name}:")
        print(f"  No data available")

# ============================================================
# SECTION 6: RANGE EXTENSION ANALYSIS
# ============================================================

print("\n\n" + "=" * 80)
print("SECTION 6: RANGE EXTENSION ANALYSIS")
print("=" * 80)
print("\nHow far do sessions extend beyond prior session ranges?")
print("Measured as % of the reference price level (normalized across all price levels)")
print("\nExample: If Asia high is 10,000 and London extends to 10,050,")
print("         that's a 0.50% extension above Asia high ((50/10,000) * 100)")

print("\n" + "-" * 80)
print("LONDON SESSION EXTENSIONS RELATIVE TO ASIA SESSION")
print("-" * 80)

london_above_asia = range_extensions['london_above_asia']
london_below_asia = range_extensions['london_below_asia']

if london_above_asia:
    median_above = pd.Series(london_above_asia).median()
    mean_above = pd.Series(london_above_asia).mean()
    print(f"\nLondon extends ABOVE Asia high:")
    print(f"  Sample size         : {len(london_above_asia)}")
    print(f"  Median extension    : {median_above:.3f}% of Asia high")
    print(f"  Mean extension      : {mean_above:.3f}% of Asia high")

if london_below_asia:
    median_below = pd.Series(london_below_asia).median()
    mean_below = pd.Series(london_below_asia).mean()
    print(f"\nLondon extends BELOW Asia low:")
    print(f"  Sample size         : {len(london_below_asia)}")
    print(f"  Median extension    : {median_below:.3f}% of Asia low")
    print(f"  Mean extension      : {mean_below:.3f}% of Asia low")

print("\n" + "-" * 80)
print("NY SESSION EXTENSIONS RELATIVE TO ASIA SESSION")
print("-" * 80)

ny_above_asia = range_extensions['ny_above_asia']
ny_below_asia = range_extensions['ny_below_asia']

if ny_above_asia:
    median_above = pd.Series(ny_above_asia).median()
    mean_above = pd.Series(ny_above_asia).mean()
    print(f"\nNY extends ABOVE Asia high:")
    print(f"  Sample size         : {len(ny_above_asia)}")
    print(f"  Median extension    : {median_above:.3f}% of Asia high")
    print(f"  Mean extension      : {mean_above:.3f}% of Asia high")

if ny_below_asia:
    median_below = pd.Series(ny_below_asia).median()
    mean_below = pd.Series(ny_below_asia).mean()
    print(f"\nNY extends BELOW Asia low:")
    print(f"  Sample size         : {len(ny_below_asia)}")
    print(f"  Median extension    : {median_below:.3f}% of Asia low")
    print(f"  Mean extension      : {mean_below:.3f}% of Asia low")

print("\n" + "-" * 80)
print("NY SESSION EXTENSIONS RELATIVE TO LONDON SESSION")
print("-" * 80)

ny_above_london = range_extensions['ny_above_london']
ny_below_london = range_extensions['ny_below_london']

if ny_above_london:
    median_above = pd.Series(ny_above_london).median()
    mean_above = pd.Series(ny_above_london).mean()
    print(f"\nNY extends ABOVE London high:")
    print(f"  Sample size         : {len(ny_above_london)}")
    print(f"  Median extension    : {median_above:.3f}% of London high")
    print(f"  Mean extension      : {mean_above:.3f}% of London high")

if ny_below_london:
    median_below = pd.Series(ny_below_london).median()
    mean_below = pd.Series(ny_below_london).mean()
    print(f"\nNY extends BELOW London low:")
    print(f"  Sample size         : {len(ny_below_london)}")
    print(f"  Median extension    : {median_below:.3f}% of London low")
    print(f"  Mean extension      : {mean_below:.3f}% of London low")

print("\n" + "-" * 80)
print("NY SESSION EXTENSIONS RELATIVE TO OVERNIGHT SESSION (Asia + London)")
print("-" * 80)

ny_above_overnight = range_extensions['ny_above_overnight']
ny_below_overnight = range_extensions['ny_below_overnight']

if ny_above_overnight:
    median_above = pd.Series(ny_above_overnight).median()
    mean_above = pd.Series(ny_above_overnight).mean()
    print(f"\nNY extends ABOVE Overnight high:")
    print(f"  Sample size         : {len(ny_above_overnight)}")
    print(f"  Median extension    : {median_above:.3f}% of Overnight high")
    print(f"  Mean extension      : {mean_above:.3f}% of Overnight high")

if ny_below_overnight:
    median_below = pd.Series(ny_below_overnight).median()
    mean_below = pd.Series(ny_below_overnight).mean()
    print(f"\nNY extends BELOW Overnight low:")
    print(f"  Sample size         : {len(ny_below_overnight)}")
    print(f"  Median extension    : {median_below:.3f}% of Overnight low")
    print(f"  Mean extension      : {mean_below:.3f}% of Overnight low")

# ============================================================
# SECTION 7: NEW - SEQUENTIAL HIT ANALYSIS FOR SESSION LEVELS
# ============================================================

print("\n\n" + "=" * 80)
print("SECTION 7: SEQUENTIAL HIT ANALYSIS - SESSION LEVELS")
print("=" * 80)
print("\nWithin each session, which prior session level is hit first?")
print("After that first hit, what is the probability of hitting other levels?")
print("\nRESULTS ARE CONDITIONAL on where the session opens relative to prior session midpoint")

# ============================================================
# PART A: LONDON SESSION ANALYSIS
# ============================================================

print("\n\n" + "=" * 80)
print("PART A: LONDON SESSION (2 AM - 8 AM)")
print("=" * 80)
print("\nWhich Asia session level is hit first during London session?")
print("Results conditioned on: London open vs Asia midpoint")

asia_level_names = {
    'asia_high': 'Asia Session High',
    'asia_low': 'Asia Session Low',
    'none': 'No Asia levels hit'
}

for position, position_label in [
    ('above_midpoint', 'LONDON OPENS ABOVE ASIA MIDPOINT'),
    ('below_midpoint', 'LONDON OPENS BELOW ASIA MIDPOINT')
]:
    print("\n\n" + "=" * 80)
    print(position_label)
    print("=" * 80)
    
    print("\n" + "-" * 80)
    print("DISTRIBUTION: First Asia level hit in London session")
    print("-" * 80)
    
    total_london_hits = sum(london_first_hit_count[position].values()) - london_first_hit_count[position]['none']
    
    if total_london_hits > 0:
        for level_key in ['asia_high', 'asia_low', 'none']:
            count = london_first_hit_count[position][level_key]
            if count > 0:
                pct = (count / (total_london_hits + london_first_hit_count[position]['none'])) * 100
                print(f"\n{asia_level_names[level_key]}:")
                print(f"  Count  : {count}")
                print(f"  %      : {pct:.2f}%")
    
    print("\n\n" + "-" * 80)
    print("SEQUENTIAL HIT RATES: After first Asia level is hit, what else gets hit?")
    print("-" * 80)
    
    for first_level_key in ['asia_high', 'asia_low']:
        if london_first_hit_count[position][first_level_key] > 0:
            print(f"\n{'=' * 60}")
            print(f"WHEN {asia_level_names[first_level_key].upper()} IS HIT FIRST:")
            print(f"Total occurrences: {london_first_hit_count[position][first_level_key]}")
            print("=" * 60)
            
            for subsequent_level_key in ['asia_high', 'asia_low']:
                if subsequent_level_key != first_level_key:
                    data = london_sequential_hits[position][first_level_key][subsequent_level_key]
                    if data['total'] > 0:
                        hit_rate = (data['hit'] / data['total']) * 100
                        print(f"\n  {asia_level_names[subsequent_level_key]}:")
                        print(f"    Times analyzed    : {data['total']}")
                        print(f"    Times hit after   : {data['hit']}")
                        print(f"    Hit rate          : {hit_rate:.2f}%")

# ============================================================
# PART B: NY SESSION ANALYSIS
# ============================================================

print("\n\n" + "=" * 80)
print("PART B: NY SESSION (8 AM - 4 PM)")
print("=" * 80)
print("\nWhich prior session level is hit first during NY session?")
print("Results conditioned on: NY open vs Overnight midpoint")

session_level_display_names = {
    'asia_high': 'Asia Session High',
    'asia_low': 'Asia Session Low',
    'london_high': 'London Session High',
    'london_low': 'London Session Low',
    'overnight_high': 'Overnight Session High',
    'overnight_low': 'Overnight Session Low',
    'none': 'No session levels hit'
}

for position, position_label in [
    ('above_midpoint', 'NY OPENS ABOVE OVERNIGHT MIDPOINT'),
    ('below_midpoint', 'NY OPENS BELOW OVERNIGHT MIDPOINT')
]:
    print("\n\n" + "=" * 80)
    print(position_label)
    print("=" * 80)
    
    print("\n" + "-" * 80)
    print("DISTRIBUTION: First session level hit in NY session")
    print("-" * 80)
    
    total_ny_hits = sum(ny_first_hit_count[position].values()) - ny_first_hit_count[position]['none']
    
    if total_ny_hits > 0:
        for level_key in ['asia_high', 'asia_low', 'london_high', 'london_low', 'overnight_high', 'overnight_low', 'none']:
            count = ny_first_hit_count[position][level_key]
            if count > 0:
                pct = (count / (total_ny_hits + ny_first_hit_count[position]['none'])) * 100
                print(f"\n{session_level_display_names[level_key]}:")
                print(f"  Count  : {count}")
                print(f"  %      : {pct:.2f}%")
    
    print("\n\n" + "-" * 80)
    print("SEQUENTIAL HIT RATES: After first session level is hit, what else gets hit?")
    print("-" * 80)
    
    for first_level_key in ['asia_high', 'asia_low', 'london_high', 'london_low', 'overnight_high', 'overnight_low']:
        if ny_first_hit_count[position][first_level_key] > 0:
            print(f"\n\n{'=' * 60}")
            print(f"WHEN {session_level_display_names[first_level_key].upper()} IS HIT FIRST:")
            print(f"Total occurrences: {ny_first_hit_count[position][first_level_key]}")
            print("=" * 60)
            
            # Sort subsequent levels by hit rate (descending)
            subsequent_levels = []
            for subsequent_level_key in ['asia_high', 'asia_low', 'london_high', 'london_low', 'overnight_high', 'overnight_low']:
                if subsequent_level_key != first_level_key:
                    data = ny_sequential_hits[position][first_level_key][subsequent_level_key]
                    if data['total'] > 0:
                        hit_rate = (data['hit'] / data['total']) * 100
                        subsequent_levels.append((session_level_display_names[subsequent_level_key], data, hit_rate))
            
            # Sort by hit rate descending
            subsequent_levels.sort(key=lambda x: x[2], reverse=True)
            
            # Print sorted results
            for level_name, data, hit_rate in subsequent_levels:
                print(f"\n  {level_name}:")
                print(f"    Times analyzed    : {data['total']}")
                print(f"    Times hit after   : {data['hit']}")
                print(f"    Hit rate          : {hit_rate:.2f}%")

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)


FUTURES ANALYSIS REPORT

Data Period: 2013-01-01 to 2025-03-14
Total trading days analyzed: 3785


SECTION 1: SESSION PATTERN STATISTICS

Days with London Engulfs Asia               : 741
Days with Asia Engulfs London               : 203
Days with London Partially Engulfs Upwards  : 1249
Days with London Partially Engulfs Downwards: 939

------------ London Engulfs Asia ------------
New York high takes London high     | 72.33%
New York low takes London low       | 71.12%
New York engulfs overnight session  | 44.13%
NY Cash Session (9:30-4:00) Bullish | 54.80%
NY Cash Session (9:30-4:00) Bearish | 45.20%

------------ Asia Engulfs London ------------
New York high takes Asia high       | 75.86%
New York low takes Asia low         | 65.52%
New York engulfs overnight session  | 44.83%
NY Cash Session (9:30-4:00) Bullish | 58.42%
NY Cash Session (9:30-4:00) Bearish | 41.58%

----- London Partially Engulfs Upwards -----
New York high takes London high     | 81.51%
New York low takes London

In [ ]:
## Modified NQ Analysis: Session patterns, midpoint analysis, and hit rate statistics

# ============================================================
# SECTION 1: SESSION PATTERN STATISTICS (KEPT FROM ORIGINAL)
# ============================================================

# Initialize dictionaries to store stats for each pattern
stats = {
    'london_engulfs_asia': {
        'total': 0,
        'ny_high_takes_london_high': 0,
        'ny_low_takes_london_low': 0,
        'ny_engulfs_overnight': 0,
        'hod_hours': defaultdict(int),
        'lod_hours': defaultdict(int),
        'ny_cash_bullish': 0,
        'ny_cash_bearish': 0
    },
    'asia_engulfs_london': {
        'total': 0,
        'ny_high_takes_asia_high': 0,
        'ny_low_takes_asia_low': 0,
        'ny_engulfs_overnight': 0,
        'hod_hours': defaultdict(int),
        'lod_hours': defaultdict(int),
        'ny_cash_bullish': 0,
        'ny_cash_bearish': 0
    },
    'london_partially_engulfs_upwards': {
        'total': 0,
        'ny_high_takes_london_high': 0,
        'ny_low_takes_london_low': 0,
        'ny_low_takes_asia_low': 0,
        'ny_engulfs_overnight': 0,
        'hod_hours': defaultdict(int),
        'lod_hours': defaultdict(int),
        'ny_cash_bullish': 0,
        'ny_cash_bearish': 0
    },
    'london_partially_engulfs_downwards': {
        'total': 0,
        'ny_low_takes_london_low': 0,
        'ny_high_takes_london_high': 0,
        'ny_high_takes_asia_high': 0,
        'ny_engulfs_overnight': 0,
        'hod_hours': defaultdict(int),
        'lod_hours': defaultdict(int),
        'ny_cash_bullish': 0,
        'ny_cash_bearish': 0
    }
}

# Track overall HOD/LOD hours
overall_hod_hours = defaultdict(int)
overall_lod_hours = defaultdict(int)
valid_days_count = 0

# ============================================================
# SECTION 2: NEW - SESSION MIDPOINT ANALYSIS
# ============================================================

# Track session open relative to prior session midpoint
midpoint_stats = {
    'london_vs_asia': {
        'above_midpoint': {'hit_high_first': 0, 'hit_low_first': 0, 'hit_neither': 0},
        'below_midpoint': {'hit_high_first': 0, 'hit_low_first': 0, 'hit_neither': 0}
    },
    'ny_vs_london': {
        'above_midpoint': {'hit_high_first': 0, 'hit_low_first': 0, 'hit_neither': 0},
        'below_midpoint': {'hit_high_first': 0, 'hit_low_first': 0, 'hit_neither': 0}
    },
    'ny_vs_overnight': {  # NY vs combined Asia+London
        'above_midpoint': {'hit_high_first': 0, 'hit_low_first': 0, 'hit_neither': 0},
        'below_midpoint': {'hit_high_first': 0, 'hit_low_first': 0, 'hit_neither': 0}
    }
}

# ============================================================
# SECTION 3: NEW - HIT RATE STATISTICS (8 AM - 4 PM)
# ============================================================

# Track hit rates during NY session (8 AM - 4 PM)
hit_rate_stats = {
    'prev_day_high': {'hit': 0, 'total': 0},
    'prev_day_low': {'hit': 0, 'total': 0},
    'seven_to_eight_midpoint': {'hit': 0, 'total': 0},
    'midnight_open': {'hit': 0, 'total': 0}
}

# Conditional hit rates based on NY open position relative to London midpoint
conditional_hit_rates = {
    'ny_above_london_mid': {
        'seven_to_eight_midpoint': {'hit': 0, 'total': 0},
        'midnight_open': {'hit': 0, 'total': 0},
        'two_to_three_midpoint': {'hit': 0, 'total': 0}
    },
    'ny_below_london_mid': {
        'seven_to_eight_midpoint': {'hit': 0, 'total': 0},
        'midnight_open': {'hit': 0, 'total': 0},
        'two_to_three_midpoint': {'hit': 0, 'total': 0}
    }
}

# Hit rates by session pattern
pattern_hit_rates = {
    'london_engulfs_asia': {
        'ny_above_london_mid': {
            'seven_to_eight_midpoint': {'hit': 0, 'total': 0},
            'midnight_open': {'hit': 0, 'total': 0},
            'two_to_three_midpoint': {'hit': 0, 'total': 0}
        },
        'ny_below_london_mid': {
            'seven_to_eight_midpoint': {'hit': 0, 'total': 0},
            'midnight_open': {'hit': 0, 'total': 0},
            'two_to_three_midpoint': {'hit': 0, 'total': 0}
        }
    },
    'asia_engulfs_london': {
        'ny_above_london_mid': {
            'seven_to_eight_midpoint': {'hit': 0, 'total': 0},
            'midnight_open': {'hit': 0, 'total': 0},
            'two_to_three_midpoint': {'hit': 0, 'total': 0}
        },
        'ny_below_london_mid': {
            'seven_to_eight_midpoint': {'hit': 0, 'total': 0},
            'midnight_open': {'hit': 0, 'total': 0},
            'two_to_three_midpoint': {'hit': 0, 'total': 0}
        }
    },
    'london_partially_engulfs_upwards': {
        'ny_above_london_mid': {
            'seven_to_eight_midpoint': {'hit': 0, 'total': 0},
            'midnight_open': {'hit': 0, 'total': 0},
            'two_to_three_midpoint': {'hit': 0, 'total': 0}
        },
        'ny_below_london_mid': {
            'seven_to_eight_midpoint': {'hit': 0, 'total': 0},
            'midnight_open': {'hit': 0, 'total': 0},
            'two_to_three_midpoint': {'hit': 0, 'total': 0}
        }
    },
    'london_partially_engulfs_downwards': {
        'ny_above_london_mid': {
            'seven_to_eight_midpoint': {'hit': 0, 'total': 0},
            'midnight_open': {'hit': 0, 'total': 0},
            'two_to_three_midpoint': {'hit': 0, 'total': 0}
        },
        'ny_below_london_mid': {
            'seven_to_eight_midpoint': {'hit': 0, 'total': 0},
            'midnight_open': {'hit': 0, 'total': 0},
            'two_to_three_midpoint': {'hit': 0, 'total': 0}
        }
    }
}

# ============================================================
# SECTION 4: NEW - DISTANCE FROM 8 AM OPEN TO KEY LEVELS
# ============================================================

# Track % distance from 8am open to each level
distance_from_8am_open = {
    'seven_to_eight_midpoint': [],
    'midnight_open': [],
    'two_to_three_midpoint': [],
    'prev_day_high': [],
    'prev_day_low': []
}

# ============================================================
# SECTION 5: NEW - RANGE EXTENSION ANALYSIS
# ============================================================

# Track how far sessions extend in terms of prior session range units
range_extensions = {
    'london_above_asia': [],
    'london_below_asia': [],
    'ny_above_asia': [],
    'ny_below_asia': [],
    'ny_above_london': [],
    'ny_below_london': [],
    'ny_above_overnight': [],
    'ny_below_overnight': []
}

# ============================================================
# SECTION 6: NEW - SEQUENTIAL HIT ANALYSIS FOR SESSION LEVELS
# ============================================================

# LONDON SESSION: Track Asia high/low hits conditional on London open vs Asia midpoint
london_sequential_hits = {
    'above_midpoint': {
        'asia_high': {
            'asia_low': {'hit': 0, 'total': 0}
        },
        'asia_low': {
            'asia_high': {'hit': 0, 'total': 0}
        }
    },
    'below_midpoint': {
        'asia_high': {
            'asia_low': {'hit': 0, 'total': 0}
        },
        'asia_low': {
            'asia_high': {'hit': 0, 'total': 0}
        }
    }
}

london_first_hit_count = {
    'above_midpoint': {
        'asia_high': 0,
        'asia_low': 0,
        'none': 0
    },
    'below_midpoint': {
        'asia_high': 0,
        'asia_low': 0,
        'none': 0
    }
}

# NY SESSION: Track ONLY London high/low, conditional on NY open vs LONDON midpoint
ny_sequential_hits = {
    'above_midpoint': {  # NY opens above LONDON midpoint
        'london_high': {
            'london_low': {'hit': 0, 'total': 0}
        },
        'london_low': {
            'london_high': {'hit': 0, 'total': 0}
        }
    },
    'below_midpoint': {  # NY opens below LONDON midpoint
        'london_high': {
            'london_low': {'hit': 0, 'total': 0}
        },
        'london_low': {
            'london_high': {'hit': 0, 'total': 0}
        }
    }
}

ny_first_hit_count = {
    'above_midpoint': {
        'london_high': 0,
        'london_low': 0,
        'none': 0
    },
    'below_midpoint': {
        'london_high': 0,
        'london_low': 0,
        'none': 0
    }
}

# ============================================================
# SECTION 8: NEW - ASIA LEVEL HIT RATES IN NY, SPLIT BY LONDON
# ============================================================

asia_levels_in_ny = {
    'asia_high': {
        'hit_in_london': {'hit_in_ny': 0, 'total': 0},
        'not_hit_in_london': {'hit_in_ny': 0, 'total': 0}
    },
    'asia_low': {
        'hit_in_london': {'hit_in_ny': 0, 'total': 0},
        'not_hit_in_london': {'hit_in_ny': 0, 'total': 0}
    }
}

# ============================================================
# MAIN ANALYSIS LOOP
# ============================================================

for date in dates[1:]:  # Start from the second day
    # Map the NY trading day back to the actual date
    actual_date = date + pd.Timedelta(hours=7)

    # Define the full NY trading day range
    ny_trading_day_start = actual_date.replace(hour=18, minute=0, second=0) - pd.Timedelta(days=1)
    ny_trading_day_end = actual_date.replace(hour=17, minute=0, second=0)

    # Get all data for this NY trading day
    full_day_data = df.loc[ny_trading_day_start:ny_trading_day_end]
    
    if full_day_data.empty:
        continue

    # Find HOD and LOD for the entire trading day
    day_high = full_day_data['High'].max()
    day_low = full_day_data['Low'].min()
    
    # Find the hour when HOD occurred
    hod_time = full_day_data[full_day_data['High'] == day_high].index[0]
    hod_hour = hod_time.hour
    
    # Find the hour when LOD occurred
    lod_time = full_day_data[full_day_data['Low'] == day_low].index[0]
    lod_hour = lod_time.hour
    
    # Track overall HOD/LOD hours
    overall_hod_hours[hod_hour] += 1
    overall_lod_hours[lod_hour] += 1
    valid_days_count += 1

    # Define session time ranges
    asian_start = actual_date.replace(hour=20, minute=0, second=0) - pd.Timedelta(days=1)
    asian_end = actual_date.replace(hour=2, minute=0, second=0)
    london_start = asian_end
    london_end = actual_date.replace(hour=8, minute=0, second=0)
    ny_start = london_end
    ny_end = actual_date.replace(hour=16, minute=0, second=0)

    # Extract session data
    asian_data = df.loc[asian_start:asian_end]
    london_data = df.loc[london_start:london_end]
    ny_data = df.loc[ny_start:ny_end]

    # Check for early closure in NY session
    if not ny_data.empty:
        last_timestamp = ny_data.index.max()
        if last_timestamp < ny_end - pd.Timedelta(minutes=30):
            ny_end = last_timestamp
            ny_data = df.loc[ny_start:ny_end]

    # Skip if any session data is missing
    if asian_data.empty or london_data.empty or ny_data.empty:
        continue

    # Calculate high and low for each session
    asian_high = asian_data['High'].max()
    asian_low = asian_data['Low'].min()
    london_high = london_data['High'].max()
    london_low = london_data['Low'].min()
    ny_high = ny_data['High'].max()
    ny_low = ny_data['Low'].min()

    # Skip if any high/low is NaN
    if pd.isna([asian_high, asian_low, london_high, london_low, ny_high, ny_low]).any():
        continue

    # Calculate overnight session range
    overnight_high = max(asian_high, london_high)
    overnight_low = min(asian_low, london_low)

    # ============================================================
    # SEQUENTIAL HIT ANALYSIS - LONDON SESSION
    # ============================================================
    
    # Determine London open position relative to Asia midpoint
    asia_midpoint = (asian_high + asian_low) / 2
    london_open = london_data['Open'].iloc[0] if not london_data.empty else None
    
    if london_open is not None:
        if london_open > asia_midpoint:
            london_position = 'above_midpoint'
        else:
            london_position = 'below_midpoint'
        
        # Track which Asia level is hit first during London session
        asia_levels_london = {
            'asia_high': asian_high,
            'asia_low': asian_low
        }
        
        asia_hit_times_london = {}
        
        # Track when each Asia level is hit during London session
        for idx, row in london_data.iterrows():
            for level_name, level_price in asia_levels_london.items():
                if level_name not in asia_hit_times_london:
                    if row['Low'] <= level_price <= row['High']:
                        asia_hit_times_london[level_name] = idx
        
        # Determine which Asia level was hit first in London session
        if asia_hit_times_london:
            first_hit_london = min(asia_hit_times_london, key=asia_hit_times_london.get)
            first_hit_time_london = asia_hit_times_london[first_hit_london]
            london_first_hit_count[london_position][first_hit_london] += 1
            
            # Check if OTHER Asia level gets hit after the first hit
            subsequent_london_data = london_data.loc[first_hit_time_london:]
            
            for level_name, level_price in asia_levels_london.items():
                if level_name != first_hit_london:
                    london_sequential_hits[london_position][first_hit_london][level_name]['total'] += 1
                    
                    # Check if this level was hit in subsequent London data
                    for idx, row in subsequent_london_data.iterrows():
                        if row['Low'] <= level_price <= row['High']:
                            london_sequential_hits[london_position][first_hit_london][level_name]['hit'] += 1
                            break
        else:
            london_first_hit_count[london_position]['none'] += 1

    # ============================================================
    # SESSION PATTERN CLASSIFICATION (ORIGINAL CODE)
    # ============================================================
    
    current_pattern = None

    # Pattern 1: London Engulfs Asia
    if london_high > asian_high and london_low < asian_low:
        current_pattern = 'london_engulfs_asia'
        stats['london_engulfs_asia']['total'] += 1
        if ny_high > london_high:
            stats['london_engulfs_asia']['ny_high_takes_london_high'] += 1
        if ny_low < london_low:
            stats['london_engulfs_asia']['ny_low_takes_london_low'] += 1
        if ny_high > overnight_high and ny_low < overnight_low:
            stats['london_engulfs_asia']['ny_engulfs_overnight'] += 1

    # Pattern 2: Asia Engulfs London
    elif london_high <= asian_high and london_low >= asian_low:
        current_pattern = 'asia_engulfs_london'
        stats['asia_engulfs_london']['total'] += 1
        if ny_high > asian_high:
            stats['asia_engulfs_london']['ny_high_takes_asia_high'] += 1
        if ny_low < asian_low:
            stats['asia_engulfs_london']['ny_low_takes_asia_low'] += 1
        if ny_high > overnight_high and ny_low < overnight_low:
            stats['asia_engulfs_london']['ny_engulfs_overnight'] += 1

    # Pattern 3: London Partially Engulfs Upwards
    elif london_high > asian_high and london_low >= asian_low:
        current_pattern = 'london_partially_engulfs_upwards'
        stats['london_partially_engulfs_upwards']['total'] += 1
        if ny_high > london_high:
            stats['london_partially_engulfs_upwards']['ny_high_takes_london_high'] += 1
        if ny_low < london_low:
            stats['london_partially_engulfs_upwards']['ny_low_takes_london_low'] += 1
        if ny_low < asian_low:
            stats['london_partially_engulfs_upwards']['ny_low_takes_asia_low'] += 1
        if ny_high > overnight_high and ny_low < overnight_low:
            stats['london_partially_engulfs_upwards']['ny_engulfs_overnight'] += 1

    # Pattern 4: London Partially Engulfs Downwards
    elif london_low < asian_low and london_high <= asian_high:
        current_pattern = 'london_partially_engulfs_downwards'
        stats['london_partially_engulfs_downwards']['total'] += 1
        if ny_low < london_low:
            stats['london_partially_engulfs_downwards']['ny_low_takes_london_low'] += 1
        if ny_high > london_high:
            stats['london_partially_engulfs_downwards']['ny_high_takes_london_high'] += 1
        if ny_high > asian_high:
            stats['london_partially_engulfs_downwards']['ny_high_takes_asia_high'] += 1
        if ny_high > overnight_high and ny_low < overnight_low:
            stats['london_partially_engulfs_downwards']['ny_engulfs_overnight'] += 1

    # Track HOD/LOD hours for this pattern
    if current_pattern:
        stats[current_pattern]['hod_hours'][hod_hour] += 1
        stats[current_pattern]['lod_hours'][lod_hour] += 1
        
        # Track NY cash session direction
        ny_cash_start = actual_date.replace(hour=9, minute=30, second=0)
        ny_cash_end = actual_date.replace(hour=16, minute=0, second=0)
        
        ny_cash_data = df.loc[ny_cash_start:ny_cash_end]
        
        if not ny_cash_data.empty:
            ny_cash_open = ny_cash_data['Open'].iloc[0]
            ny_cash_close = ny_cash_data['Close'].iloc[-1]
            
            if ny_cash_close > ny_cash_open:
                stats[current_pattern]['ny_cash_bullish'] += 1
            elif ny_cash_close < ny_cash_open:
                stats[current_pattern]['ny_cash_bearish'] += 1

    # ============================================================
    # NEW: SESSION MIDPOINT ANALYSIS
    # ============================================================
    
    # 1. London open vs Asia midpoint
    asia_midpoint = (asian_high + asian_low) / 2
    london_open = london_data['Open'].iloc[0] if not london_data.empty else None
    
    if london_open is not None:
        if london_open > asia_midpoint:
            position = 'above_midpoint'
        else:
            position = 'below_midpoint'
        
        # Check which is hit first during London session: Asia high or Asia low
        hit_high_first = False
        hit_low_first = False
        
        for idx, row in london_data.iterrows():
            if not hit_high_first and not hit_low_first:
                if row['High'] >= asian_high:
                    hit_high_first = True
                    midpoint_stats['london_vs_asia'][position]['hit_high_first'] += 1
                    break
                elif row['Low'] <= asian_low:
                    hit_low_first = True
                    midpoint_stats['london_vs_asia'][position]['hit_low_first'] += 1
                    break
        
        if not hit_high_first and not hit_low_first:
            midpoint_stats['london_vs_asia'][position]['hit_neither'] += 1
    
    # 2. NY open vs London midpoint
    london_midpoint = (london_high + london_low) / 2
    ny_open = ny_data['Open'].iloc[0] if not ny_data.empty else None
    
    if ny_open is not None:
        if ny_open > london_midpoint:
            position = 'above_midpoint'
        else:
            position = 'below_midpoint'
        
        # Check which is hit first during NY session: London high or London low
        hit_high_first = False
        hit_low_first = False
        
        for idx, row in ny_data.iterrows():
            if not hit_high_first and not hit_low_first:
                if row['High'] >= london_high:
                    hit_high_first = True
                    midpoint_stats['ny_vs_london'][position]['hit_high_first'] += 1
                    break
                elif row['Low'] <= london_low:
                    hit_low_first = True
                    midpoint_stats['ny_vs_london'][position]['hit_low_first'] += 1
                    break
        
        if not hit_high_first and not hit_low_first:
            midpoint_stats['ny_vs_london'][position]['hit_neither'] += 1
    
    # 3. NY open vs Overnight (Asia + London) midpoint
    overnight_midpoint = (overnight_high + overnight_low) / 2
    
    if ny_open is not None:
        if ny_open > overnight_midpoint:
            position = 'above_midpoint'
        else:
            position = 'below_midpoint'
        
        # Check which is hit first during NY session: Overnight high or Overnight low
        hit_high_first = False
        hit_low_first = False
        
        for idx, row in ny_data.iterrows():
            if not hit_high_first and not hit_low_first:
                if row['High'] >= overnight_high:
                    hit_high_first = True
                    midpoint_stats['ny_vs_overnight'][position]['hit_high_first'] += 1
                    break
                elif row['Low'] <= overnight_low:
                    hit_low_first = True
                    midpoint_stats['ny_vs_overnight'][position]['hit_low_first'] += 1
                    break
        
        if not hit_high_first and not hit_low_first:
            midpoint_stats['ny_vs_overnight'][position]['hit_neither'] += 1

    # ============================================================
    # NEW: RANGE EXTENSION ANALYSIS
    # ============================================================
    
    # Calculate range extensions as % of the reference price level
    
    # London extensions relative to Asia
    if asian_high > 0:
        if london_high > asian_high:
            london_above_asia_pct = ((london_high - asian_high) / asian_high) * 100
            range_extensions['london_above_asia'].append(london_above_asia_pct)
        else:
            range_extensions['london_above_asia'].append(0)
    
    if asian_low > 0:
        if london_low < asian_low:
            london_below_asia_pct = ((asian_low - london_low) / asian_low) * 100
            range_extensions['london_below_asia'].append(london_below_asia_pct)
        else:
            range_extensions['london_below_asia'].append(0)
    
    # NY extensions relative to Asia
    if asian_high > 0:
        if ny_high > asian_high:
            ny_above_asia_pct = ((ny_high - asian_high) / asian_high) * 100
            range_extensions['ny_above_asia'].append(ny_above_asia_pct)
        else:
            range_extensions['ny_above_asia'].append(0)
    
    if asian_low > 0:
        if ny_low < asian_low:
            ny_below_asia_pct = ((asian_low - ny_low) / asian_low) * 100
            range_extensions['ny_below_asia'].append(ny_below_asia_pct)
        else:
            range_extensions['ny_below_asia'].append(0)
    
    # NY extensions relative to London
    if london_high > 0:
        if ny_high > london_high:
            ny_above_london_pct = ((ny_high - london_high) / london_high) * 100
            range_extensions['ny_above_london'].append(ny_above_london_pct)
        else:
            range_extensions['ny_above_london'].append(0)
    
    if london_low > 0:
        if ny_low < london_low:
            ny_below_london_pct = ((london_low - ny_low) / london_low) * 100
            range_extensions['ny_below_london'].append(ny_below_london_pct)
        else:
            range_extensions['ny_below_london'].append(0)
    
    # NY extensions relative to Overnight
    if overnight_high > 0:
        if ny_high > overnight_high:
            ny_above_overnight_pct = ((ny_high - overnight_high) / overnight_high) * 100
            range_extensions['ny_above_overnight'].append(ny_above_overnight_pct)
        else:
            range_extensions['ny_above_overnight'].append(0)
    
    if overnight_low > 0:
        if ny_low < overnight_low:
            ny_below_overnight_pct = ((overnight_low - ny_low) / overnight_low) * 100
            range_extensions['ny_below_overnight'].append(ny_below_overnight_pct)
        else:
            range_extensions['ny_below_overnight'].append(0)

    # ============================================================
    # NEW: HIT RATE ANALYSIS (8 AM - 4 PM)
    # ============================================================
    
    # Define the analysis window (8 AM - 4 PM)
    analysis_start = actual_date.replace(hour=8, minute=0, second=0)
    analysis_end = actual_date.replace(hour=16, minute=0, second=0)
    analysis_data = df.loc[analysis_start:analysis_end]
    
    if not analysis_data.empty:
        # Get 8 AM open price for distance calculations
        eight_am_open = analysis_data['Open'].iloc[0]
        analysis_high = analysis_data['High'].max()
        analysis_low = analysis_data['Low'].min()
        
        # Determine NY open position relative to London midpoint
        london_midpoint = (london_high + london_low) / 2
        ny_open = ny_data['Open'].iloc[0] if not ny_data.empty else None
        
        if ny_open is not None:
            if ny_open > london_midpoint:
                ny_position = 'ny_above_london_mid'
            else:
                ny_position = 'ny_below_london_mid'
        else:
            ny_position = None
        
        # ============================================================
        # CALCULATE INTRADAY PRICE LEVELS
        # ============================================================
        
        # 1. Previous day high/low
        prev_date = date - pd.Timedelta(days=1)
        prev_day_actual_date = prev_date + pd.Timedelta(hours=7)
        prev_day_start = prev_day_actual_date.replace(hour=18, minute=0, second=0) - pd.Timedelta(days=1)
        prev_day_end = prev_day_actual_date.replace(hour=17, minute=0, second=0)
        
        prev_day_data = df.loc[prev_day_start:prev_day_end]
        
        if not prev_day_data.empty:
            prev_day_high = prev_day_data['High'].max()
            prev_day_low = prev_day_data['Low'].min()
        else:
            prev_day_high = None
            prev_day_low = None
        
        # 2. 7-8 AM hourly midpoint
        seven_to_eight_start = actual_date.replace(hour=7, minute=0, second=0)
        seven_to_eight_end = actual_date.replace(hour=8, minute=0, second=0)
        seven_to_eight_data = df.loc[seven_to_eight_start:seven_to_eight_end]
        
        if not seven_to_eight_data.empty:
            seven_to_eight_high = seven_to_eight_data['High'].max()
            seven_to_eight_low = seven_to_eight_data['Low'].min()
            seven_to_eight_midpoint = (seven_to_eight_high + seven_to_eight_low) / 2
        else:
            seven_to_eight_midpoint = None
        
        # 3. Midnight open
        midnight_time = actual_date.replace(hour=0, minute=0, second=0)
        midnight_data = df.loc[midnight_time:]
        
        if not midnight_data.empty:
            midnight_open = midnight_data['Open'].iloc[0]
        else:
            midnight_open = None
        
        # 4. 2-3 AM hourly midpoint
        two_to_three_start = actual_date.replace(hour=2, minute=0, second=0)
        two_to_three_end = actual_date.replace(hour=3, minute=0, second=0)
        two_to_three_data = df.loc[two_to_three_start:two_to_three_end]
        
        if not two_to_three_data.empty:
            two_to_three_high = two_to_three_data['High'].max()
            two_to_three_low = two_to_three_data['Low'].min()
            two_to_three_midpoint = (two_to_three_high + two_to_three_low) / 2
        else:
            two_to_three_midpoint = None
        
        # ============================================================
        # SEQUENTIAL HIT ANALYSIS - NY SESSION (FIXED)
        # NOW USES LONDON MIDPOINT AND ONLY TRACKS LONDON LEVELS
        # ============================================================
        
        # Determine NY open position relative to LONDON midpoint
        london_midpoint = (london_high + london_low) / 2
        ny_open = ny_data['Open'].iloc[0] if not ny_data.empty else None
        
        if ny_open is not None:
            if ny_open > london_midpoint:
                ny_position_midpoint = 'above_midpoint'
            else:
                ny_position_midpoint = 'below_midpoint'
            
            # Define ONLY London levels (not Asia, not Overnight)
            ny_session_levels = {
                'london_high': london_high,
                'london_low': london_low
            }
            
            # Dictionary to track when each London level is hit during NY session
            session_level_hit_times = {}
            
            # Iterate through NY session (8 AM - 4 PM) and track when each London level is hit
            for idx, row in analysis_data.iterrows():
                for level_name, level_price in ny_session_levels.items():
                    # If level not yet hit and price touched this bar, record the time
                    if level_name not in session_level_hit_times:
                        if row['Low'] <= level_price <= row['High']:
                            session_level_hit_times[level_name] = idx
            
            # Determine which London level was hit first
            if session_level_hit_times:
                first_hit_level = min(session_level_hit_times, key=session_level_hit_times.get)
                first_hit_time = session_level_hit_times[first_hit_level]
                ny_first_hit_count[ny_position_midpoint][first_hit_level] += 1
                
                # Now check if the OTHER London level got hit AFTER the first hit
                subsequent_data = analysis_data.loc[first_hit_time:]
                
                for level_name, level_price in ny_session_levels.items():
                    if level_name != first_hit_level:  # Only track the OTHER London level
                        ny_sequential_hits[ny_position_midpoint][first_hit_level][level_name]['total'] += 1
                        
                        # Check if this level was hit in the subsequent data
                        for idx, row in subsequent_data.iterrows():
                            if row['Low'] <= level_price <= row['High']:
                                ny_sequential_hits[ny_position_midpoint][first_hit_level][level_name]['hit'] += 1
                                break
            else:
                ny_first_hit_count[ny_position_midpoint]['none'] += 1

        # ============================================================
        # NEW: ASIA LEVEL HIT RATES IN NY, SPLIT BY LONDON HIT STATUS
        # ============================================================

        # Was Asia high hit during London session?
        asia_high_hit_in_london = any(
            row['Low'] <= asian_high <= row['High']
            for _, row in london_data.iterrows()
        )

        # Was Asia low hit during London session?
        asia_low_hit_in_london = any(
            row['Low'] <= asian_low <= row['High']
            for _, row in london_data.iterrows()
        )

        # Was Asia high hit during NY session (8 AM - 4 PM)?
        asia_high_hit_in_ny = any(
            row['Low'] <= asian_high <= row['High']
            for _, row in analysis_data.iterrows()
        )

        # Was Asia low hit during NY session?
        asia_low_hit_in_ny = any(
            row['Low'] <= asian_low <= row['High']
            for _, row in analysis_data.iterrows()
        )

        # Record results for Asia high
        london_key_high = 'hit_in_london' if asia_high_hit_in_london else 'not_hit_in_london'
        asia_levels_in_ny['asia_high'][london_key_high]['total'] += 1
        if asia_high_hit_in_ny:
            asia_levels_in_ny['asia_high'][london_key_high]['hit_in_ny'] += 1

        # Record results for Asia low
        london_key_low = 'hit_in_london' if asia_low_hit_in_london else 'not_hit_in_london'
        asia_levels_in_ny['asia_low'][london_key_low]['total'] += 1
        if asia_low_hit_in_ny:
            asia_levels_in_ny['asia_low'][london_key_low]['hit_in_ny'] += 1

        # ============================================================
        # ORIGINAL HIT RATE TRACKING (UNCHANGED)
        # ============================================================
        
        if not prev_day_data.empty:
            hit_rate_stats['prev_day_high']['total'] += 1
            hit_rate_stats['prev_day_low']['total'] += 1
            
            if analysis_high >= prev_day_high:
                hit_rate_stats['prev_day_high']['hit'] += 1
            if analysis_low <= prev_day_low:
                hit_rate_stats['prev_day_low']['hit'] += 1
            
            # Track % distance from 8am open to prev day levels
            if eight_am_open > 0:
                pct_dist_to_high = ((prev_day_high - eight_am_open) / eight_am_open) * 100
                pct_dist_to_low = ((prev_day_low - eight_am_open) / eight_am_open) * 100
                distance_from_8am_open['prev_day_high'].append(pct_dist_to_high)
                distance_from_8am_open['prev_day_low'].append(pct_dist_to_low)
        
        if not seven_to_eight_data.empty:
            hit_rate_stats['seven_to_eight_midpoint']['total'] += 1
            
            # Check if price touched the midpoint during analysis window
            hit_seven_eight = analysis_low <= seven_to_eight_midpoint <= analysis_high
            if hit_seven_eight:
                hit_rate_stats['seven_to_eight_midpoint']['hit'] += 1
            
            # Track % distance from 8am open to 7-8am midpoint
            if eight_am_open > 0:
                pct_dist_to_midpoint = ((seven_to_eight_midpoint - eight_am_open) / eight_am_open) * 100
                distance_from_8am_open['seven_to_eight_midpoint'].append(pct_dist_to_midpoint)
            
            # Conditional tracking based on NY position
            if ny_position is not None:
                conditional_hit_rates[ny_position]['seven_to_eight_midpoint']['total'] += 1
                if hit_seven_eight:
                    conditional_hit_rates[ny_position]['seven_to_eight_midpoint']['hit'] += 1
                
                # Pattern-specific tracking
                if current_pattern is not None:
                    pattern_hit_rates[current_pattern][ny_position]['seven_to_eight_midpoint']['total'] += 1
                    if hit_seven_eight:
                        pattern_hit_rates[current_pattern][ny_position]['seven_to_eight_midpoint']['hit'] += 1
        
        if not midnight_data.empty:
            hit_rate_stats['midnight_open']['total'] += 1
            
            # Check if price touched the midnight open during analysis window
            hit_midnight = analysis_low <= midnight_open <= analysis_high
            if hit_midnight:
                hit_rate_stats['midnight_open']['hit'] += 1
            
            # Track % distance from 8am open to midnight open
            if eight_am_open > 0:
                pct_dist_to_midnight = ((midnight_open - eight_am_open) / eight_am_open) * 100
                distance_from_8am_open['midnight_open'].append(pct_dist_to_midnight)
            
            # Conditional tracking based on NY position
            if ny_position is not None:
                conditional_hit_rates[ny_position]['midnight_open']['total'] += 1
                if hit_midnight:
                    conditional_hit_rates[ny_position]['midnight_open']['hit'] += 1
                
                # Pattern-specific tracking
                if current_pattern is not None:
                    pattern_hit_rates[current_pattern][ny_position]['midnight_open']['total'] += 1
                    if hit_midnight:
                        pattern_hit_rates[current_pattern][ny_position]['midnight_open']['hit'] += 1
        
        if not two_to_three_data.empty:
            # Track % distance from 8am open to 2-3am midpoint
            if eight_am_open > 0:
                pct_dist_to_two_three = ((two_to_three_midpoint - eight_am_open) / eight_am_open) * 100
                distance_from_8am_open['two_to_three_midpoint'].append(pct_dist_to_two_three)
            
            # Conditional tracking based on NY position
            if ny_position is not None:
                hit_two_three = analysis_low <= two_to_three_midpoint <= analysis_high
                
                conditional_hit_rates[ny_position]['two_to_three_midpoint']['total'] += 1
                if hit_two_three:
                    conditional_hit_rates[ny_position]['two_to_three_midpoint']['hit'] += 1
                
                # Pattern-specific tracking
                if current_pattern is not None:
                    pattern_hit_rates[current_pattern][ny_position]['two_to_three_midpoint']['total'] += 1
                    if hit_two_three:
                        pattern_hit_rates[current_pattern][ny_position]['two_to_three_midpoint']['hit'] += 1

# ============================================================
# CALCULATE PERCENTAGES
# ============================================================

# Pattern stats percentages (original)
for pattern_key in stats.keys():
    if stats[pattern_key]['total'] > 0:
        pattern = stats[pattern_key]
        total = pattern['total']
        
        for key in list(pattern.keys()):
            if key.endswith('_pct'):
                continue
            if key in ['total', 'hod_hours', 'lod_hours', 'ny_cash_bullish', 'ny_cash_bearish']:
                continue
            
            pct_key = key + '_pct'
            stats[pattern_key][pct_key] = round(pattern[key] / total * 100, 2)


# ============================================================
# PRINT RESULTS - SECTION 7 (SEQUENTIAL HIT ANALYSIS)
# ============================================================

print("\n\n" + "=" * 80)
print("SECTION 7: SEQUENTIAL HIT ANALYSIS - SESSION LEVELS")
print("=" * 80)
print("\n*** FIXED: NY session now uses LONDON MIDPOINT (not Overnight) ***")
print("*** FIXED: NY session now ONLY tracks London High/Low (not all levels) ***")

# ============================================================
# PART A: LONDON SESSION ANALYSIS
# ============================================================

print("\n\n" + "=" * 80)
print("PART A: LONDON SESSION (2 AM - 8 AM)")
print("=" * 80)
print("\nWhich Asia session level is hit first during London session?")
print("Results conditioned on: London open vs Asia midpoint")

asia_level_names = {
    'asia_high': 'Asia Session High',
    'asia_low': 'Asia Session Low',
    'none': 'No Asia levels hit'
}

for position, position_label in [
    ('above_midpoint', 'LONDON OPENS ABOVE ASIA MIDPOINT'),
    ('below_midpoint', 'LONDON OPENS BELOW ASIA MIDPOINT')
]:
    print("\n\n" + "=" * 80)
    print(position_label)
    print("=" * 80)
    
    print("\n" + "-" * 80)
    print("DISTRIBUTION: First Asia level hit in London session")
    print("-" * 80)
    
    total_london_hits = sum(london_first_hit_count[position].values()) - london_first_hit_count[position]['none']
    
    if total_london_hits > 0:
        for level_key in ['asia_high', 'asia_low', 'none']:
            count = london_first_hit_count[position][level_key]
            if count > 0:
                pct = (count / (total_london_hits + london_first_hit_count[position]['none'])) * 100
                print(f"\n{asia_level_names[level_key]}:")
                print(f"  Count  : {count}")
                print(f"  %      : {pct:.2f}%")
    
    print("\n\n" + "-" * 80)
    print("SEQUENTIAL HIT RATES: After first Asia level is hit, what else gets hit?")
    print("-" * 80)
    
    for first_level_key in ['asia_high', 'asia_low']:
        if london_first_hit_count[position][first_level_key] > 0:
            print(f"\n{'=' * 60}")
            print(f"WHEN {asia_level_names[first_level_key].upper()} IS HIT FIRST:")
            print(f"Total occurrences: {london_first_hit_count[position][first_level_key]}")
            print("=" * 60)
            
            for subsequent_level_key in ['asia_high', 'asia_low']:
                if subsequent_level_key != first_level_key:
                    data = london_sequential_hits[position][first_level_key][subsequent_level_key]
                    if data['total'] > 0:
                        hit_rate = (data['hit'] / data['total']) * 100
                        print(f"\n  {asia_level_names[subsequent_level_key]}:")
                        print(f"    Times analyzed    : {data['total']}")
                        print(f"    Times hit after   : {data['hit']}")
                        print(f"    Hit rate          : {hit_rate:.2f}%")

# ============================================================
# PART B: NY SESSION ANALYSIS (FIXED)
# ============================================================

print("\n\n" + "=" * 80)
print("PART B: NY SESSION (8 AM - 4 PM) *** FIXED ***")
print("=" * 80)
print("\nWhich London level is hit first during NY session?")
print("Results conditioned on: NY open vs LONDON midpoint (NOT Overnight)")

london_level_names = {
    'london_high': 'London Session High',
    'london_low': 'London Session Low',
    'none': 'No London levels hit'
}

for position, position_label in [
    ('above_midpoint', 'NY OPENS ABOVE LONDON MIDPOINT'),
    ('below_midpoint', 'NY OPENS BELOW LONDON MIDPOINT')
]:
    print("\n\n" + "=" * 80)
    print(position_label)
    print("=" * 80)
    
    print("\n" + "-" * 80)
    print("DISTRIBUTION: First London level hit in NY session")
    print("-" * 80)
    
    total_ny_hits = sum(ny_first_hit_count[position].values()) - ny_first_hit_count[position]['none']
    
    if total_ny_hits > 0:
        for level_key in ['london_high', 'london_low', 'none']:
            count = ny_first_hit_count[position][level_key]
            if count > 0:
                pct = (count / (total_ny_hits + ny_first_hit_count[position]['none'])) * 100
                print(f"\n{london_level_names[level_key]}:")
                print(f"  Count  : {count}")
                print(f"  %      : {pct:.2f}%")
    
    print("\n\n" + "-" * 80)
    print("SEQUENTIAL HIT RATES: After first London level is hit, does the OTHER one get hit?")
    print("-" * 80)
    
    for first_level_key in ['london_high', 'london_low']:
        if ny_first_hit_count[position][first_level_key] > 0:
            print(f"\n{'=' * 60}")
            print(f"WHEN {london_level_names[first_level_key].upper()} IS HIT FIRST:")
            print(f"Total occurrences: {ny_first_hit_count[position][first_level_key]}")
            print("=" * 60)
            
            for subsequent_level_key in ['london_high', 'london_low']:
                if subsequent_level_key != first_level_key:
                    data = ny_sequential_hits[position][first_level_key][subsequent_level_key]
                    if data['total'] > 0:
                        hit_rate = (data['hit'] / data['total']) * 100
                        print(f"\n  {london_level_names[subsequent_level_key]}:")
                        print(f"    Times analyzed    : {data['total']}")
                        print(f"    Times hit after   : {data['hit']}")
                        print(f"    Hit rate          : {hit_rate:.2f}%")

# ============================================================
# PRINT RESULTS - SECTION 8 (ASIA LEVELS IN NY BY LONDON STATUS)
# ============================================================

print("\n\n" + "=" * 80)
print("SECTION 8: ASIA LEVEL HIT RATES DURING NY SESSION (8 AM - 4 PM)")
print("Split by whether the level was already hit during London session (2 AM - 8 AM)")
print("=" * 80)

level_labels = {'asia_high': 'ASIA HIGH', 'asia_low': 'ASIA LOW'}

for level_key, label in level_labels.items():
    print(f"\n{label}")
    print("-" * 40)
    for condition in ['hit_in_london', 'not_hit_in_london']:
        data = asia_levels_in_ny[level_key][condition]
        if data['total'] > 0:
            rate = (data['hit_in_ny'] / data['total']) * 100
            cond_label = "Already hit in London" if condition == 'hit_in_london' else "NOT hit in London"
            print(f"  {cond_label}:")
            print(f"    Total days : {data['total']}")
            print(f"    Hit in NY  : {data['hit_in_ny']}")
            print(f"    Hit rate   : {rate:.2f}%")

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)
print("\nNOTE: Run this script to get the CORRECTED NY sequential statistics")
print("These stats should then be used to update the TradingView indicator")

In [ ]:
## Enhanced Opening Range and Initial Balance Analysis - Conditional Combinations

# ============================================================
# ENHANCED DATA STRUCTURES WITH CONDITIONAL COMBINATIONS
# ============================================================

def create_orb_combo_dict():
    """Create nested dictionary for ORB conditional statistics"""
    return {
        'bullish': {
            'high_first': {'high_breaks': 0, 'low_breaks': 0, 'neither': 0, 'midpoint_retest': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0},
            'low_first': {'high_breaks': 0, 'low_breaks': 0, 'neither': 0, 'midpoint_retest': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0},
            'high_breaks': {'midpoint_retest': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0},
            'low_breaks': {'midpoint_retest': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0}
        },
        'bearish': {
            'high_first': {'high_breaks': 0, 'low_breaks': 0, 'neither': 0, 'midpoint_retest': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0},
            'low_first': {'high_breaks': 0, 'low_breaks': 0, 'neither': 0, 'midpoint_retest': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0},
            'high_breaks': {'midpoint_retest': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0},
            'low_breaks': {'midpoint_retest': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0}
        }
    }

orb_combos = {
    '5min': create_orb_combo_dict(),
    '15min': create_orb_combo_dict()
}

# IB combination statistics with midpoint retest
def create_ib_combo_dict():
    """Create dictionary for IB combo tracking including general midpoint retest"""
    return {
        'high_breaks': 0, 
        'low_breaks': 0, 
        'neither': 0, 
        'bullish_close': 0, 
        'bearish_close': 0,
        'midpoint_retest_any': 0,  # NEW: Retest regardless of break
        'midpoint_retest_times': [],  # NEW: Time to retest in minutes
        'total': 0
    }

ib_combos = {
    'bullish': {
        'high_first': {
            'close_above_mid': create_ib_combo_dict(),
            'close_below_mid': create_ib_combo_dict()
        },
        'low_first': {
            'close_above_mid': create_ib_combo_dict(),
            'close_below_mid': create_ib_combo_dict()
        },
        'close_above_mid': create_ib_combo_dict(),
        'close_below_mid': create_ib_combo_dict()
    },
    'bearish': {
        'high_first': {
            'close_above_mid': create_ib_combo_dict(),
            'close_below_mid': create_ib_combo_dict()
        },
        'low_first': {
            'close_above_mid': create_ib_combo_dict(),
            'close_below_mid': create_ib_combo_dict()
        },
        'close_above_mid': create_ib_combo_dict(),
        'close_below_mid': create_ib_combo_dict()
    }
}

# IB Midpoint Retest Statistics (after break)
def create_retest_dict():
    """Create dictionary for midpoint retest tracking AFTER breaks"""
    return {
        'total_breaks': 0,
        'midpoint_retested': 0,
        'retest_confirms': 0,      # Bounces back in original direction
        'retest_reverses': 0,      # Breaks through to opposite side
        'retest_stays_range': 0    # Stays between midpoint and opposite extreme
    }

ib_midpoint_retest = {
    'overall': {
        'high_breaks_first': create_retest_dict(),
        'low_breaks_first': create_retest_dict()
    },
    'by_direction': {
        'bullish': {
            'high_breaks_first': create_retest_dict(),
            'low_breaks_first': create_retest_dict()
        },
        'bearish': {
            'high_breaks_first': create_retest_dict(),
            'low_breaks_first': create_retest_dict()
        }
    },
    'by_close_position': {
        'close_above_mid': {
            'high_breaks_first': create_retest_dict(),
            'low_breaks_first': create_retest_dict()
        },
        'close_below_mid': {
            'high_breaks_first': create_retest_dict(),
            'low_breaks_first': create_retest_dict()
        }
    },
    'by_direction_and_close': {
        'bullish': {
            'close_above_mid': {
                'high_breaks_first': create_retest_dict(),
                'low_breaks_first': create_retest_dict()
            },
            'close_below_mid': {
                'high_breaks_first': create_retest_dict(),
                'low_breaks_first': create_retest_dict()
            }
        },
        'bearish': {
            'close_above_mid': {
                'high_breaks_first': create_retest_dict(),
                'low_breaks_first': create_retest_dict()
            },
            'close_below_mid': {
                'high_breaks_first': create_retest_dict(),
                'low_breaks_first': create_retest_dict()
            }
        }
    }
}

# Extension tracking with better categorization
extension_stats = {
    '5min': {
        'above_high': {'extended': [], 'no_extension': 0},
        'below_low': {'extended': [], 'no_extension': 0}
    },
    '15min': {
        'above_high': {'extended': [], 'no_extension': 0},
        'below_low': {'extended': [], 'no_extension': 0}
    },
    'ib': {
        'above_high': {'extended': [], 'no_extension': 0},
        'below_low': {'extended': [], 'no_extension': 0}
    }
}

# ============================================================
# MAIN ANALYSIS LOOP
# ============================================================

for date in dates[1:]:
    actual_date = date + pd.Timedelta(hours=7)
    
    # Define NY cash session (9:30 AM - 4:00 PM)
    ny_cash_start = actual_date.replace(hour=9, minute=30, second=0)
    ny_cash_end = actual_date.replace(hour=16, minute=0, second=0)
    
    ny_cash_data = df.loc[ny_cash_start:ny_cash_end]
    
    if ny_cash_data.empty:
        continue
    
    # Check for early closure
    last_timestamp = ny_cash_data.index.max()
    if last_timestamp < ny_cash_end - pd.Timedelta(minutes=30):
        ny_cash_end = last_timestamp
        ny_cash_data = df.loc[ny_cash_start:ny_cash_end]
    
    # Get NY cash session metrics
    ny_cash_open = ny_cash_data['Open'].iloc[0]
    ny_cash_close = ny_cash_data['Close'].iloc[-1]
    ny_cash_high = ny_cash_data['High'].max()
    ny_cash_low = ny_cash_data['Low'].min()
    ny_cash_bullish = ny_cash_close > ny_cash_open
    
    # ============================================================
    # OPENING RANGE COMBINATION ANALYSIS
    # ============================================================
    
    for orb_minutes, orb_key in [(5, '5min'), (15, '15min')]:
        orb_start = ny_cash_start
        orb_end = ny_cash_start + pd.Timedelta(minutes=orb_minutes)
        orb_data = df.loc[orb_start:orb_end]
        
        if orb_data.empty:
            continue
        
        orb_open = orb_data['Open'].iloc[0]
        orb_close = orb_data['Close'].iloc[-1]
        orb_high = orb_data['High'].max()
        orb_low = orb_data['Low'].min()
        orb_midpoint = (orb_high + orb_low) / 2
        
        if pd.isna([orb_open, orb_close, orb_high, orb_low]).any():
            continue
        
        # Determine ORB characteristics
        orb_direction = 'bullish' if orb_close > orb_open else 'bearish'
        
        # Which extreme formed first
        orb_extreme_first = None
        for idx, row in orb_data.iterrows():
            if row['High'] >= orb_high:
                orb_extreme_first = 'high_first'
                break
            elif row['Low'] <= orb_low:
                orb_extreme_first = 'low_first'
                break
        
        if orb_extreme_first is None:
            continue
        
        # Which side breaks first after ORB
        post_orb_data = df.loc[orb_end:ny_cash_end]
        first_break = None
        high_break_time = None
        low_break_time = None
        midpoint_retest = False
        
        if not post_orb_data.empty:
            high_broken = False
            low_broken = False
            
            for idx, row in post_orb_data.iterrows():
                if not high_broken and row['High'] > orb_high:
                    high_broken = True
                    high_break_time = idx
                if not low_broken and row['Low'] < orb_low:
                    low_broken = True
                    low_break_time = idx
                
                if high_broken and low_broken:
                    break
            
            if high_broken and low_broken:
                first_break = 'high_breaks' if high_break_time < low_break_time else 'low_breaks'
            elif high_broken:
                first_break = 'high_breaks'
            elif low_broken:
                first_break = 'low_breaks'
            else:
                first_break = 'neither'
            
            # Check for midpoint retest after break
            if first_break == 'high_breaks':
                post_break_data = df.loc[high_break_time:ny_cash_end]
                for idx, row in post_break_data.iterrows():
                    if row['Low'] <= orb_midpoint:
                        midpoint_retest = True
                        break
            elif first_break == 'low_breaks':
                post_break_data = df.loc[low_break_time:ny_cash_end]
                for idx, row in post_break_data.iterrows():
                    if row['High'] >= orb_midpoint:
                        midpoint_retest = True
                        break
        else:
            first_break = 'neither'
        
        # Track combinations: ORB direction × Extreme first → Break direction, Close direction, Midpoint retest
        combo = orb_combos[orb_key][orb_direction][orb_extreme_first]
        combo['total'] += 1
        
        if first_break == 'high_breaks':
            combo['high_breaks'] += 1
        elif first_break == 'low_breaks':
            combo['low_breaks'] += 1
        else:
            combo['neither'] += 1
        
        if midpoint_retest:
            combo['midpoint_retest'] += 1
        
        if ny_cash_bullish:
            combo['bullish_close'] += 1
        else:
            combo['bearish_close'] += 1
        
        # Track combinations: ORB direction × Break direction → Midpoint retest, Close direction
        if first_break in ['high_breaks', 'low_breaks']:
            break_combo = orb_combos[orb_key][orb_direction][first_break]
            break_combo['total'] += 1
            
            if midpoint_retest:
                break_combo['midpoint_retest'] += 1
            
            if ny_cash_bullish:
                break_combo['bullish_close'] += 1
            else:
                break_combo['bearish_close'] += 1
        
        # Extension tracking
        if orb_high > 0:
            if ny_cash_high > orb_high:
                extension_pct = ((ny_cash_high - orb_high) / orb_high) * 100
                extension_stats[orb_key]['above_high']['extended'].append(extension_pct)
            else:
                extension_stats[orb_key]['above_high']['no_extension'] += 1
        
        if orb_low > 0:
            if ny_cash_low < orb_low:
                extension_pct = ((orb_low - ny_cash_low) / orb_low) * 100
                extension_stats[orb_key]['below_low']['extended'].append(extension_pct)
            else:
                extension_stats[orb_key]['below_low']['no_extension'] += 1
    
    # ============================================================
    # INITIAL BALANCE COMBINATION ANALYSIS
    # ============================================================
    
    ib_start = ny_cash_start
    ib_end = ny_cash_start + pd.Timedelta(hours=1)
    ib_data = df.loc[ib_start:ib_end]
    
    if ib_data.empty:
        continue
    
    ib_open = ib_data['Open'].iloc[0]
    ib_close = ib_data['Close'].iloc[-1]
    ib_high = ib_data['High'].max()
    ib_low = ib_data['Low'].min()
    ib_midpoint = (ib_high + ib_low) / 2
    
    if pd.isna([ib_open, ib_close, ib_high, ib_low]).any():
        continue
    
    # Determine IB characteristics
    ib_direction = 'bullish' if ib_close > ib_open else 'bearish'
    ib_close_position = 'close_above_mid' if ib_close > ib_midpoint else 'close_below_mid'
    
    # Which extreme formed first
    ib_extreme_first = None
    for idx, row in ib_data.iterrows():
        if row['High'] >= ib_high:
            ib_extreme_first = 'high_first'
            break
        elif row['Low'] <= ib_low:
            ib_extreme_first = 'low_first'
            break
    
    if ib_extreme_first is None:
        continue
    
    # ============================================================
    # NEW: GENERAL MIDPOINT RETEST (Regardless of Break)
    # ============================================================
    
    post_ib_data = df.loc[ib_end:ny_cash_end]
    
    general_midpoint_retested = False
    time_to_general_retest = None
    
    if not post_ib_data.empty:
        # Find when price first moves away from midpoint (beyond 0.1% threshold)
        threshold = ib_midpoint * 0.001  # 0.1% threshold
        left_midpoint = False
        left_midpoint_time = None
        
        for idx, row in post_ib_data.iterrows():
            if not left_midpoint:
                if abs(row['Close'] - ib_midpoint) > threshold:
                    left_midpoint = True
                    left_midpoint_time = idx
            else:
                # Check if price comes back to midpoint
                if abs(row['Low'] - ib_midpoint) <= threshold or abs(row['High'] - ib_midpoint) <= threshold:
                    general_midpoint_retested = True
                    time_to_general_retest = (idx - left_midpoint_time).total_seconds() / 60
                    break
    
    # ============================================================
    # Which side breaks first after IB
    # ============================================================
    
    first_break = None
    
    if not post_ib_data.empty:
        high_broken = False
        low_broken = False
        high_break_time = None
        low_break_time = None
        
        for idx, row in post_ib_data.iterrows():
            if not high_broken and row['High'] > ib_high:
                high_broken = True
                high_break_time = idx
            if not low_broken and row['Low'] < ib_low:
                low_broken = True
                low_break_time = idx
            
            if high_broken and low_broken:
                break
        
        if high_broken and low_broken:
            first_break = 'high_breaks' if high_break_time < low_break_time else 'low_breaks'
        elif high_broken:
            first_break = 'high_breaks'
        elif low_broken:
            first_break = 'low_breaks'
        else:
            first_break = 'neither'
        
        # ============================================================
        # MIDPOINT RETEST ANALYSIS AFTER BREAK
        # ============================================================
        
        def update_retest_stats(stats_dict, break_type):
            """Update midpoint retest statistics for a given break type"""
            stats_dict[break_type]['total_breaks'] += 1
            
            if break_type == 'high_breaks_first':
                # Check if midpoint was retested after high break
                post_break_data = df.loc[high_break_time:ny_cash_end]
                midpoint_retested = False
                retest_time = None
                
                for idx, row in post_break_data.iterrows():
                    if row['Low'] <= ib_midpoint:
                        midpoint_retested = True
                        retest_time = idx
                        stats_dict[break_type]['midpoint_retested'] += 1
                        break
                
                if midpoint_retested and retest_time:
                    # Check what happened after retest
                    post_retest_data = df.loc[retest_time:ny_cash_end]
                    
                    confirmed = False
                    reversed = False
                    
                    for idx, row in post_retest_data.iterrows():
                        # Check if it bounced back and confirmed high break
                        if row['High'] > ib_high:
                            confirmed = True
                            stats_dict[break_type]['retest_confirms'] += 1
                            break
                        # Check if it reversed and broke low
                        if row['Low'] < ib_low:
                            reversed = True
                            stats_dict[break_type]['retest_reverses'] += 1
                            break
                    
                    # If neither confirmed nor reversed, it stayed in range
                    if not confirmed and not reversed:
                        stats_dict[break_type]['retest_stays_range'] += 1
            
            elif break_type == 'low_breaks_first':
                # Check if midpoint was retested after low break
                post_break_data = df.loc[low_break_time:ny_cash_end]
                midpoint_retested = False
                retest_time = None
                
                for idx, row in post_break_data.iterrows():
                    if row['High'] >= ib_midpoint:
                        midpoint_retested = True
                        retest_time = idx
                        stats_dict[break_type]['midpoint_retested'] += 1
                        break
                
                if midpoint_retested and retest_time:
                    # Check what happened after retest
                    post_retest_data = df.loc[retest_time:ny_cash_end]
                    
                    confirmed = False
                    reversed = False
                    
                    for idx, row in post_retest_data.iterrows():
                        # Check if it bounced back and confirmed low break
                        if row['Low'] < ib_low:
                            confirmed = True
                            stats_dict[break_type]['retest_confirms'] += 1
                            break
                        # Check if it reversed and broke high
                        if row['High'] > ib_high:
                            reversed = True
                            stats_dict[break_type]['retest_reverses'] += 1
                            break
                    
                    # If neither confirmed nor reversed, it stayed in range
                    if not confirmed and not reversed:
                        stats_dict[break_type]['retest_stays_range'] += 1
        
        # Update all relevant stat categories
        if first_break == 'high_breaks':
            update_retest_stats(ib_midpoint_retest['overall'], 'high_breaks_first')
            update_retest_stats(ib_midpoint_retest['by_direction'][ib_direction], 'high_breaks_first')
            update_retest_stats(ib_midpoint_retest['by_close_position'][ib_close_position], 'high_breaks_first')
            update_retest_stats(ib_midpoint_retest['by_direction_and_close'][ib_direction][ib_close_position], 'high_breaks_first')
        
        elif first_break == 'low_breaks':
            update_retest_stats(ib_midpoint_retest['overall'], 'low_breaks_first')
            update_retest_stats(ib_midpoint_retest['by_direction'][ib_direction], 'low_breaks_first')
            update_retest_stats(ib_midpoint_retest['by_close_position'][ib_close_position], 'low_breaks_first')
            update_retest_stats(ib_midpoint_retest['by_direction_and_close'][ib_direction][ib_close_position], 'low_breaks_first')
    
    else:
        first_break = 'neither'
    
    # ============================================================
    # Track combinations with GENERAL midpoint retest
    # ============================================================
    
    # IB direction × Extreme first × Close position
    combo = ib_combos[ib_direction][ib_extreme_first][ib_close_position]
    combo['total'] += 1
    
    if first_break == 'high_breaks':
        combo['high_breaks'] += 1
    elif first_break == 'low_breaks':
        combo['low_breaks'] += 1
    else:
        combo['neither'] += 1
    
    if ny_cash_bullish:
        combo['bullish_close'] += 1
    else:
        combo['bearish_close'] += 1
    
    # NEW: Track general midpoint retest
    if general_midpoint_retested:
        combo['midpoint_retest_any'] += 1
    if time_to_general_retest is not None:
        combo['midpoint_retest_times'].append(time_to_general_retest)
    
    # IB direction × Close position
    close_combo = ib_combos[ib_direction][ib_close_position]
    close_combo['total'] += 1
    
    if first_break == 'high_breaks':
        close_combo['high_breaks'] += 1
    elif first_break == 'low_breaks':
        close_combo['low_breaks'] += 1
    else:
        close_combo['neither'] += 1
    
    if ny_cash_bullish:
        close_combo['bullish_close'] += 1
    else:
        close_combo['bearish_close'] += 1
    
    # NEW: Track general midpoint retest
    if general_midpoint_retested:
        close_combo['midpoint_retest_any'] += 1
    if time_to_general_retest is not None:
        close_combo['midpoint_retest_times'].append(time_to_general_retest)
    
    # Extension tracking
    if ib_high > 0:
        if ny_cash_high > ib_high:
            extension_pct = ((ny_cash_high - ib_high) / ib_high) * 100
            extension_stats['ib']['above_high']['extended'].append(extension_pct)
        else:
            extension_stats['ib']['above_high']['no_extension'] += 1
    
    if ib_low > 0:
        if ny_cash_low < ib_low:
            extension_pct = ((ib_low - ny_cash_low) / ib_low) * 100
            extension_stats['ib']['below_low']['extended'].append(extension_pct)
        else:
            extension_stats['ib']['below_low']['no_extension'] += 1

# ============================================================
# PRINT RESULTS
# ============================================================

def print_combo_stats(combo, indent="  "):
    """Print combination statistics with percentages including general midpoint retest"""
    if combo['total'] == 0:
        print(f"{indent}No data")
        return
    
    total = combo['total']
    print(f"{indent}Total occurrences: {total}")
    
    # Break direction
    if 'high_breaks' in combo:
        high_pct = (combo['high_breaks'] / total) * 100
        low_pct = (combo['low_breaks'] / total) * 100
        neither_pct = (combo['neither'] / total) * 100
        print(f"{indent}  Break HIGH first : {combo['high_breaks']:4d} ({high_pct:5.1f}%)")
        print(f"{indent}  Break LOW first  : {combo['low_breaks']:4d} ({low_pct:5.1f}%)")
        print(f"{indent}  Neither breaks   : {combo['neither']:4d} ({neither_pct:5.1f}%)")
    
    # NY cash session close direction (9:30-4pm outcome)
    bull_pct = (combo['bullish_close'] / total) * 100
    bear_pct = (combo['bearish_close'] / total) * 100
    print(f"{indent}  → NY Session (9:30-4pm) closes BULLISH: {combo['bullish_close']:4d} ({bull_pct:5.1f}%)")
    print(f"{indent}  → NY Session (9:30-4pm) closes BEARISH: {combo['bearish_close']:4d} ({bear_pct:5.1f}%)")
    
    # Midpoint retest (if applicable)
    if 'midpoint_retest' in combo:
        retest_pct = (combo['midpoint_retest'] / total) * 100
        print(f"{indent}  Midpoint retest  : {combo['midpoint_retest']:4d} ({retest_pct:5.1f}%)")
    
    # NEW: General midpoint retest (regardless of break)
    if 'midpoint_retest_any' in combo:
        retest_any_pct = (combo['midpoint_retest_any'] / total) * 100
        print(f"{indent}  Midpoint retested (any): {combo['midpoint_retest_any']:4d} ({retest_any_pct:5.1f}%)")
        
        if combo['midpoint_retest_times']:
            times = combo['midpoint_retest_times']
            mean_time = np.mean(times)
            median_time = np.median(times)
            print(f"{indent}    Mean time to retest  : {mean_time:6.1f} minutes")
            print(f"{indent}    Median time to retest: {median_time:6.1f} minutes")

def print_retest_stats(stats, break_label, indent="  "):
    """Print midpoint retest statistics AFTER breaks"""
    total = stats['total_breaks']
    
    if total == 0:
        print(f"{indent}No {break_label} data")
        return
    
    print(f"\n{indent}{break_label}:")
    print(f"{indent}  Total occurrences: {total}")
    
    retested = stats['midpoint_retested']
    retest_pct = (retested / total) * 100
    no_retest_pct = 100 - retest_pct
    
    print(f"{indent}  Midpoint RETESTED     : {retested:4d} ({retest_pct:5.1f}%)")
    print(f"{indent}  Midpoint NOT retested : {total - retested:4d} ({no_retest_pct:5.1f}%)")
    
    if retested > 0:
        print(f"\n{indent}  When midpoint IS retested:")
        
        confirms = stats['retest_confirms']
        reverses = stats['retest_reverses']
        stays = stats['retest_stays_range']
        
        confirm_pct = (confirms / retested) * 100
        reverse_pct = (reverses / retested) * 100
        stays_pct = (stays / retested) * 100
        
        if 'high_breaks' in break_label.lower():
            print(f"{indent}    Bounces back UP (confirms): {confirms:4d} ({confirm_pct:5.1f}%)")
            print(f"{indent}    Reverses & breaks LOW     : {reverses:4d} ({reverse_pct:5.1f}%)")
            print(f"{indent}    Stays in range            : {stays:4d} ({stays_pct:5.1f}%)")
        else:
            print(f"{indent}    Bounces back DOWN (confirms): {confirms:4d} ({confirm_pct:5.1f}%)")
            print(f"{indent}    Reverses & breaks HIGH      : {reverses:4d} ({reverse_pct:5.1f}%)")
            print(f"{indent}    Stays in range              : {stays:4d} ({stays_pct:5.1f}%)")

def print_extension_stats(stats, label):
    """Print extension statistics showing % that extend vs don't"""
    extended = stats['extended']
    no_extension = stats['no_extension']
    total = len(extended) + no_extension
    
    if total == 0:
        print(f"  No data for {label}")
        return
    
    extension_pct = (len(extended) / total) * 100
    no_extension_pct = (no_extension / total) * 100
    
    print(f"\n  {label}:")
    print(f"    Total sessions   : {total}")
    print(f"    Sessions that extend      : {len(extended):4d} ({extension_pct:5.1f}%)")
    print(f"    Sessions with NO extension: {no_extension:4d} ({no_extension_pct:5.1f}%)")
    
    if extended:
        arr = np.array(extended)
        print(f"\n    For sessions that DO extend:")
        print(f"      Mean           : {arr.mean():.3f}%")
        print(f"      Median (50th)  : {np.percentile(arr, 50):.3f}%")
        print(f"      25th percentile: {np.percentile(arr, 25):.3f}%")
        print(f"      75th percentile: {np.percentile(arr, 75):.3f}%")
        print(f"      90th percentile: {np.percentile(arr, 90):.3f}%")
        print(f"      95th percentile: {np.percentile(arr, 95):.3f}%")
        print(f"      99th percentile: {np.percentile(arr, 99):.3f}%")

print("\n" + "=" * 80)
print("ENHANCED ORB & IB ANALYSIS: CONDITIONAL COMBINATIONS")
print("=" * 80)
print(f"\nData Period: {start_date.date()} to {df.index.max().date()}")

# ============================================================
# SECTION 1: 5-MINUTE ORB COMBINATIONS
# ============================================================

print("\n\n" + "=" * 80)
print("SECTION 1: 5-MINUTE ORB CONDITIONAL COMBINATIONS")
print("=" * 80)

for direction in ['bullish', 'bearish']:
    direction_label = direction.upper()
    print(f"\n\n{'=' * 80}")
    print(f"When 5-min ORB is {direction_label}")
    print(f"{'=' * 80}")
    
    for extreme in ['high_first', 'low_first']:
        extreme_label = "HIGH forms first" if extreme == 'high_first' else "LOW forms first"
        print(f"\n{'-' * 80}")
        print(f"Condition: ORB {direction_label} + {extreme_label}")
        print(f"{'-' * 80}")
        print_combo_stats(orb_combos['5min'][direction][extreme], "  ")
    
    print(f"\n\n{'-' * 80}")
    print(f"When ORB is {direction_label}, breakdown by which side breaks first:")
    print(f"{'-' * 80}")
    
    for break_dir in ['high_breaks', 'low_breaks']:
        break_label = "HIGH breaks first" if break_dir == 'high_breaks' else "LOW breaks first"
        print(f"\n  Condition: ORB {direction_label} + {break_label}")
        print_combo_stats(orb_combos['5min'][direction][break_dir], "    ")

# ============================================================
# SECTION 2: 15-MINUTE ORB COMBINATIONS
# ============================================================

print("\n\n" + "=" * 80)
print("SECTION 2: 15-MINUTE ORB CONDITIONAL COMBINATIONS")
print("=" * 80)

for direction in ['bullish', 'bearish']:
    direction_label = direction.upper()
    print(f"\n\n{'=' * 80}")
    print(f"When 15-min ORB is {direction_label}")
    print(f"{'=' * 80}")
    
    for extreme in ['high_first', 'low_first']:
        extreme_label = "HIGH forms first" if extreme == 'high_first' else "LOW forms first"
        print(f"\n{'-' * 80}")
        print(f"Condition: ORB {direction_label} + {extreme_label}")
        print(f"{'-' * 80}")
        print_combo_stats(orb_combos['15min'][direction][extreme], "  ")
    
    print(f"\n\n{'-' * 80}")
    print(f"When ORB is {direction_label}, breakdown by which side breaks first:")
    print(f"{'-' * 80}")
    
    for break_dir in ['high_breaks', 'low_breaks']:
        break_label = "HIGH breaks first" if break_dir == 'high_breaks' else "LOW breaks first"
        print(f"\n  Condition: ORB {direction_label} + {break_label}")
        print_combo_stats(orb_combos['15min'][direction][break_dir], "    ")

# ============================================================
# SECTION 3: INITIAL BALANCE COMBINATIONS
# ============================================================

print("\n\n" + "=" * 80)
print("SECTION 3: INITIAL BALANCE CONDITIONAL COMBINATIONS")
print("=" * 80)

for direction in ['bullish', 'bearish']:
    direction_label = direction.upper()
    print(f"\n\n{'=' * 80}")
    print(f"When IB is {direction_label}")
    print(f"{'=' * 80}")
    
    # Extreme first × Close position
    for extreme in ['high_first', 'low_first']:
        extreme_label = "HIGH forms first" if extreme == 'high_first' else "LOW forms first"
        
        for close_pos in ['close_above_mid', 'close_below_mid']:
            close_label = "closes ABOVE midpoint" if close_pos == 'close_above_mid' else "closes BELOW midpoint"
            
            print(f"\n{'-' * 80}")
            print(f"Condition: IB {direction_label} + {extreme_label} + IB {close_label}")
            print(f"{'-' * 80}")
            print_combo_stats(ib_combos[direction][extreme][close_pos], "  ")
    
    # Just close position (simpler view)
    print(f"\n\n{'-' * 80}")
    print(f"When IB is {direction_label}, breakdown by IB close position:")
    print(f"{'-' * 80}")
    
    for close_pos in ['close_above_mid', 'close_below_mid']:
        close_label = "IB closes ABOVE midpoint" if close_pos == 'close_above_mid' else "IB closes BELOW midpoint"
        
        print(f"\n  Condition: IB {direction_label} + {close_label}")
        print_combo_stats(ib_combos[direction][close_pos], "    ")

# ============================================================
# SECTION 3A: IB MIDPOINT RETEST ANALYSIS (AFTER BREAK)
# ============================================================

print("\n\n" + "=" * 80)
print("SECTION 3A: INITIAL BALANCE MIDPOINT RETEST ANALYSIS (AFTER BREAK)")
print("=" * 80)
print("\nWhat happens AFTER the IB breaks and the midpoint is retested?")

print("\n" + "-" * 80)
print("OVERALL MIDPOINT RETEST PATTERNS (All IB Sessions)")
print("-" * 80)

print_retest_stats(ib_midpoint_retest['overall']['high_breaks_first'], "When IB HIGH breaks first", "  ")
print_retest_stats(ib_midpoint_retest['overall']['low_breaks_first'], "When IB LOW breaks first", "  ")

print("\n\n" + "-" * 80)
print("MIDPOINT RETEST BY IB DIRECTION")
print("-" * 80)

for direction in ['bullish', 'bearish']:
    direction_label = direction.upper()
    print(f"\n{'~' * 80}")
    print(f"When IB is {direction_label}:")
    print(f"{'~' * 80}")
    
    print_retest_stats(ib_midpoint_retest['by_direction'][direction]['high_breaks_first'], 
                      "When HIGH breaks first", "  ")
    print_retest_stats(ib_midpoint_retest['by_direction'][direction]['low_breaks_first'], 
                      "When LOW breaks first", "  ")

print("\n\n" + "-" * 80)
print("MIDPOINT RETEST BY IB CLOSE POSITION")
print("-" * 80)

for close_pos in ['close_above_mid', 'close_below_mid']:
    close_label = "IB CLOSES ABOVE MIDPOINT" if close_pos == 'close_above_mid' else "IB CLOSES BELOW MIDPOINT"
    print(f"\n{'~' * 80}")
    print(f"When {close_label}:")
    print(f"{'~' * 80}")
    
    print_retest_stats(ib_midpoint_retest['by_close_position'][close_pos]['high_breaks_first'], 
                      "When HIGH breaks first", "  ")
    print_retest_stats(ib_midpoint_retest['by_close_position'][close_pos]['low_breaks_first'], 
                      "When LOW breaks first", "  ")

print("\n\n" + "-" * 80)
print("MIDPOINT RETEST BY IB DIRECTION AND CLOSE POSITION (Most Granular)")
print("-" * 80)

for direction in ['bullish', 'bearish']:
    direction_label = direction.upper()
    
    for close_pos in ['close_above_mid', 'close_below_mid']:
        close_label = "closes ABOVE mid" if close_pos == 'close_above_mid' else "closes BELOW mid"
        
        print(f"\n{'~' * 80}")
        print(f"IB is {direction_label} + {close_label}:")
        print(f"{'~' * 80}")
        
        print_retest_stats(ib_midpoint_retest['by_direction_and_close'][direction][close_pos]['high_breaks_first'], 
                          "When HIGH breaks first", "  ")
        print_retest_stats(ib_midpoint_retest['by_direction_and_close'][direction][close_pos]['low_breaks_first'], 
                          "When LOW breaks first", "  ")

# ============================================================
# SECTION 4: EXTENSION STATISTICS (ENHANCED)
# ============================================================

print("\n\n" + "=" * 80)
print("SECTION 4: RANGE EXTENSIONS (WITH NO-EXTENSION TRACKING)")
print("=" * 80)

print("\n" + "-" * 80)
print("5-MINUTE ORB EXTENSIONS")
print("-" * 80)
print_extension_stats(extension_stats['5min']['above_high'], "Extensions ABOVE 5-min ORB high")
print_extension_stats(extension_stats['5min']['below_low'], "Extensions BELOW 5-min ORB low")

print("\n\n" + "-" * 80)
print("15-MINUTE ORB EXTENSIONS")
print("-" * 80)
print_extension_stats(extension_stats['15min']['above_high'], "Extensions ABOVE 15-min ORB high")
print_extension_stats(extension_stats['15min']['below_low'], "Extensions BELOW 15-min ORB low")

print("\n\n" + "-" * 80)
print("INITIAL BALANCE EXTENSIONS")
print("-" * 80)
print_extension_stats(extension_stats['ib']['above_high'], "Extensions ABOVE IB high")
print_extension_stats(extension_stats['ib']['below_low'], "Extensions BELOW IB low")

# ============================================================
# SECTION 5: PREDICTIVE POWER SUMMARY
# ============================================================

print("\n\n" + "=" * 80)
print("SECTION 5: KEY PREDICTIVE PATTERNS FOR NY SESSION CLOSE (9:30-4PM)")
print("=" * 80)
print("\nHighest probability setups for predicting session close direction:")

print("\n" + "-" * 80)
print("5-MINUTE ORB STRONGEST SIGNALS")
print("-" * 80)

# Find strongest signals from 5min ORB
strongest_5min = []
for direction in ['bullish', 'bearish']:
    for extreme in ['high_first', 'low_first']:
        combo = orb_combos['5min'][direction][extreme]
        if combo['total'] >= 50:  # Minimum sample size
            bull_pct = (combo['bullish_close'] / combo['total']) * 100
            bear_pct = (combo['bearish_close'] / combo['total']) * 100
            strongest_5min.append({
                'condition': f"5min ORB {direction.upper()} + {extreme.replace('_', ' ').upper()}",
                'bull_pct': bull_pct,
                'bear_pct': bear_pct,
                'total': combo['total'],
                'dominant': 'BULLISH' if bull_pct > bear_pct else 'BEARISH',
                'edge': max(bull_pct, bear_pct)
            })

strongest_5min.sort(key=lambda x: x['edge'], reverse=True)

for i, signal in enumerate(strongest_5min[:5], 1):
    print(f"\n{i}. {signal['condition']}")
    print(f"   Sample size: {signal['total']}")
    print(f"   → Session closes {signal['dominant']}: {signal['edge']:.1f}%")
    print(f"   → Session closes {'BEARISH' if signal['dominant'] == 'BULLISH' else 'BULLISH'}: {100 - signal['edge']:.1f}%")

print("\n" + "-" * 80)
print("15-MINUTE ORB STRONGEST SIGNALS")
print("-" * 80)

# Find strongest signals from 15min ORB
strongest_15min = []
for direction in ['bullish', 'bearish']:
    for extreme in ['high_first', 'low_first']:
        combo = orb_combos['15min'][direction][extreme]
        if combo['total'] >= 50:
            bull_pct = (combo['bullish_close'] / combo['total']) * 100
            bear_pct = (combo['bearish_close'] / combo['total']) * 100
            strongest_15min.append({
                'condition': f"15min ORB {direction.upper()} + {extreme.replace('_', ' ').upper()}",
                'bull_pct': bull_pct,
                'bear_pct': bear_pct,
                'total': combo['total'],
                'dominant': 'BULLISH' if bull_pct > bear_pct else 'BEARISH',
                'edge': max(bull_pct, bear_pct)
            })

strongest_15min.sort(key=lambda x: x['edge'], reverse=True)

for i, signal in enumerate(strongest_15min[:5], 1):
    print(f"\n{i}. {signal['condition']}")
    print(f"   Sample size: {signal['total']}")
    print(f"   → Session closes {signal['dominant']}: {signal['edge']:.1f}%")
    print(f"   → Session closes {'BEARISH' if signal['dominant'] == 'BULLISH' else 'BULLISH'}: {100 - signal['edge']:.1f}%")

print("\n" + "-" * 80)
print("INITIAL BALANCE STRONGEST SIGNALS")
print("-" * 80)

# Find strongest signals from IB
strongest_ib = []
for direction in ['bullish', 'bearish']:
    for close_pos in ['close_above_mid', 'close_below_mid']:
        combo = ib_combos[direction][close_pos]
        if combo['total'] >= 50:
            bull_pct = (combo['bullish_close'] / combo['total']) * 100
            bear_pct = (combo['bearish_close'] / combo['total']) * 100
            close_label = "closes ABOVE midpoint" if close_pos == 'close_above_mid' else "closes BELOW midpoint"
            strongest_ib.append({
                'condition': f"IB {direction.upper()} + {close_label}",
                'bull_pct': bull_pct,
                'bear_pct': bear_pct,
                'total': combo['total'],
                'dominant': 'BULLISH' if bull_pct > bear_pct else 'BEARISH',
                'edge': max(bull_pct, bear_pct)
            })

strongest_ib.sort(key=lambda x: x['edge'], reverse=True)

for i, signal in enumerate(strongest_ib[:5], 1):
    print(f"\n{i}. {signal['condition']}")
    print(f"   Sample size: {signal['total']}")
    print(f"   → Session closes {signal['dominant']}: {signal['edge']:.1f}%")
    print(f"   → Session closes {'BEARISH' if signal['dominant'] == 'BULLISH' else 'BULLISH'}: {100 - signal['edge']:.1f}%")

print("\n\n" + "-" * 80)
print("MIDPOINT RETEST STRONGEST SIGNALS (After Break)")
print("-" * 80)

# Find strongest midpoint retest signals
retest_signals = []

for direction in ['bullish', 'bearish']:
    for close_pos in ['close_above_mid', 'close_below_mid']:
        for break_type in ['high_breaks_first', 'low_breaks_first']:
            stats = ib_midpoint_retest['by_direction_and_close'][direction][close_pos][break_type]
            
            if stats['midpoint_retested'] >= 20:  # Minimum sample for retests
                confirm_pct = (stats['retest_confirms'] / stats['midpoint_retested']) * 100
                reverse_pct = (stats['retest_reverses'] / stats['midpoint_retested']) * 100
                
                close_label = "closes ABOVE mid" if close_pos == 'close_above_mid' else "closes BELOW mid"
                break_label = "HIGH breaks" if break_type == 'high_breaks_first' else "LOW breaks"
                
                retest_signals.append({
                    'condition': f"IB {direction.upper()} + {close_label} + {break_label} + Midpoint retested",
                    'confirm_pct': confirm_pct,
                    'reverse_pct': reverse_pct,
                    'total': stats['midpoint_retested'],
                    'dominant': 'CONFIRMS' if confirm_pct > reverse_pct else 'REVERSES',
                    'edge': max(confirm_pct, reverse_pct)
                })

retest_signals.sort(key=lambda x: x['edge'], reverse=True)

for i, signal in enumerate(retest_signals[:8], 1):
    print(f"\n{i}. {signal['condition']}")
    print(f"   Sample size: {signal['total']} midpoint retests")
    print(f"   → {signal['dominant']}: {signal['edge']:.1f}%")
    if signal['dominant'] == 'CONFIRMS':
        print(f"   → REVERSES: {signal['reverse_pct']:.1f}%")
    else:
        print(f"   → CONFIRMS: {signal['confirm_pct']:.1f}%")

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)

In [ ]:
## Enhanced Opening Range and Initial Balance Analysis - Conditional Combinations

# ============================================================
# ENHANCED DATA STRUCTURES WITH CONDITIONAL COMBINATIONS
# ============================================================

def create_orb_combo_dict():
    """Create nested dictionary for ORB conditional statistics"""
    return {
        'bullish': {
            'high_first': {'high_breaks': 0, 'low_breaks': 0, 'neither': 0, 'midpoint_retest': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0},
            'low_first': {'high_breaks': 0, 'low_breaks': 0, 'neither': 0, 'midpoint_retest': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0},
            'high_breaks': {'midpoint_retest': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0},
            'low_breaks': {'midpoint_retest': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0}
        },
        'bearish': {
            'high_first': {'high_breaks': 0, 'low_breaks': 0, 'neither': 0, 'midpoint_retest': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0},
            'low_first': {'high_breaks': 0, 'low_breaks': 0, 'neither': 0, 'midpoint_retest': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0},
            'high_breaks': {'midpoint_retest': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0},
            'low_breaks': {'midpoint_retest': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0}
        }
    }

orb_combos = {
    '5min': create_orb_combo_dict(),
    '15min': create_orb_combo_dict()
}

# IB combination statistics
ib_combos = {
    'bullish': {
        'high_first': {
            'close_above_mid': {'high_breaks': 0, 'low_breaks': 0, 'neither': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0},
            'close_below_mid': {'high_breaks': 0, 'low_breaks': 0, 'neither': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0}
        },
        'low_first': {
            'close_above_mid': {'high_breaks': 0, 'low_breaks': 0, 'neither': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0},
            'close_below_mid': {'high_breaks': 0, 'low_breaks': 0, 'neither': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0}
        },
        'close_above_mid': {'high_breaks': 0, 'low_breaks': 0, 'neither': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0},
        'close_below_mid': {'high_breaks': 0, 'low_breaks': 0, 'neither': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0}
    },
    'bearish': {
        'high_first': {
            'close_above_mid': {'high_breaks': 0, 'low_breaks': 0, 'neither': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0},
            'close_below_mid': {'high_breaks': 0, 'low_breaks': 0, 'neither': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0}
        },
        'low_first': {
            'close_above_mid': {'high_breaks': 0, 'low_breaks': 0, 'neither': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0},
            'close_below_mid': {'high_breaks': 0, 'low_breaks': 0, 'neither': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0}
        },
        'close_above_mid': {'high_breaks': 0, 'low_breaks': 0, 'neither': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0},
        'close_below_mid': {'high_breaks': 0, 'low_breaks': 0, 'neither': 0, 'bullish_close': 0, 'bearish_close': 0, 'total': 0}
    }
}

# Extension tracking with better categorization
extension_stats = {
    '5min': {
        'above_high': {'extended': [], 'no_extension': 0},
        'below_low': {'extended': [], 'no_extension': 0}
    },
    '15min': {
        'above_high': {'extended': [], 'no_extension': 0},
        'below_low': {'extended': [], 'no_extension': 0}
    },
    'ib': {
        'above_high': {'extended': [], 'no_extension': 0},
        'below_low': {'extended': [], 'no_extension': 0}
    }
}

# NEW: ORB → IB simple tracking
orb_ib_tracking = {
    '5min': [],
    '15min': []
}

# ============================================================
# MAIN ANALYSIS LOOP
# ============================================================

for date in dates[1:]:
    actual_date = date + pd.Timedelta(hours=7)
    
    # Define NY cash session (9:30 AM - 4:00 PM)
    ny_cash_start = actual_date.replace(hour=9, minute=30, second=0)
    ny_cash_end = actual_date.replace(hour=16, minute=0, second=0)
    
    ny_cash_data = df.loc[ny_cash_start:ny_cash_end]
    
    if ny_cash_data.empty:
        continue
    
    # Check for early closure
    last_timestamp = ny_cash_data.index.max()
    if last_timestamp < ny_cash_end - pd.Timedelta(minutes=30):
        ny_cash_end = last_timestamp
        ny_cash_data = df.loc[ny_cash_start:ny_cash_end]
    
    # Get NY cash session metrics
    ny_cash_open = ny_cash_data['Open'].iloc[0]
    ny_cash_close = ny_cash_data['Close'].iloc[-1]
    ny_cash_high = ny_cash_data['High'].max()
    ny_cash_low = ny_cash_data['Low'].min()
    ny_cash_bullish = ny_cash_close > ny_cash_open
    
    # ============================================================
    # INITIAL BALANCE ANALYSIS (MOVED UP SO WE HAVE IB DATA FOR ORB)
    # ============================================================
    
    ib_start = ny_cash_start
    ib_end = ny_cash_start + pd.Timedelta(hours=1)
    ib_data = df.loc[ib_start:ib_end]
    
    if ib_data.empty:
        continue
    
    ib_open = ib_data['Open'].iloc[0]
    ib_close = ib_data['Close'].iloc[-1]
    ib_high = ib_data['High'].max()
    ib_low = ib_data['Low'].min()
    ib_midpoint = (ib_high + ib_low) / 2
    
    if pd.isna([ib_open, ib_close, ib_high, ib_low]).any():
        continue
    
    # Determine IB characteristics
    ib_direction = 'bullish' if ib_close > ib_open else 'bearish'
    ib_close_position = 'close_above_mid' if ib_close > ib_midpoint else 'close_below_mid'
    
    # Which extreme formed first
    ib_extreme_first = None
    for idx, row in ib_data.iterrows():
        if row['High'] >= ib_high:
            ib_extreme_first = 'high_first'
            break
        elif row['Low'] <= ib_low:
            ib_extreme_first = 'low_first'
            break
    
    if ib_extreme_first is None:
        continue
    
    # Which side breaks first after IB
    post_ib_data = df.loc[ib_end:ny_cash_end]
    ib_first_break = None
    
    if not post_ib_data.empty:
        high_broken = False
        low_broken = False
        high_break_time = None
        low_break_time = None
        
        for idx, row in post_ib_data.iterrows():
            if not high_broken and row['High'] > ib_high:
                high_broken = True
                high_break_time = idx
            if not low_broken and row['Low'] < ib_low:
                low_broken = True
                low_break_time = idx
            
            if high_broken and low_broken:
                break
        
        if high_broken and low_broken:
            ib_first_break = 'high_breaks' if high_break_time < low_break_time else 'low_breaks'
        elif high_broken:
            ib_first_break = 'high_breaks'
        elif low_broken:
            ib_first_break = 'low_breaks'
        else:
            ib_first_break = 'neither'
    else:
        ib_first_break = 'neither'
    
    # Track combinations: IB direction × Extreme first × Close position → Break direction, Session close
    combo = ib_combos[ib_direction][ib_extreme_first][ib_close_position]
    combo['total'] += 1
    
    if ib_first_break == 'high_breaks':
        combo['high_breaks'] += 1
    elif ib_first_break == 'low_breaks':
        combo['low_breaks'] += 1
    else:
        combo['neither'] += 1
    
    if ny_cash_bullish:
        combo['bullish_close'] += 1
    else:
        combo['bearish_close'] += 1
    
    # Track combinations: IB direction × Close position → Break direction, Session close
    close_combo = ib_combos[ib_direction][ib_close_position]
    close_combo['total'] += 1
    
    if ib_first_break == 'high_breaks':
        close_combo['high_breaks'] += 1
    elif ib_first_break == 'low_breaks':
        close_combo['low_breaks'] += 1
    else:
        close_combo['neither'] += 1
    
    if ny_cash_bullish:
        close_combo['bullish_close'] += 1
    else:
        close_combo['bearish_close'] += 1
    
    # Extension tracking
    if ib_high > 0:
        if ny_cash_high > ib_high:
            extension_pct = ((ny_cash_high - ib_high) / ib_high) * 100
            extension_stats['ib']['above_high']['extended'].append(extension_pct)
        else:
            extension_stats['ib']['above_high']['no_extension'] += 1
    
    if ib_low > 0:
        if ny_cash_low < ib_low:
            extension_pct = ((ib_low - ny_cash_low) / ib_low) * 100
            extension_stats['ib']['below_low']['extended'].append(extension_pct)
        else:
            extension_stats['ib']['below_low']['no_extension'] += 1
    
    # ============================================================
    # OPENING RANGE COMBINATION ANALYSIS
    # ============================================================
    
    for orb_minutes, orb_key in [(5, '5min'), (15, '15min')]:
        orb_start = ny_cash_start
        orb_end = ny_cash_start + pd.Timedelta(minutes=orb_minutes)
        orb_data = df.loc[orb_start:orb_end]
        
        if orb_data.empty:
            continue
        
        orb_open = orb_data['Open'].iloc[0]
        orb_close = orb_data['Close'].iloc[-1]
        orb_high = orb_data['High'].max()
        orb_low = orb_data['Low'].min()
        orb_midpoint = (orb_high + orb_low) / 2
        
        if pd.isna([orb_open, orb_close, orb_high, orb_low]).any():
            continue
        
        # Determine ORB characteristics
        orb_direction = 'bullish' if orb_close > orb_open else 'bearish'
        
        # Which extreme formed first
        orb_extreme_first = None
        for idx, row in orb_data.iterrows():
            if row['High'] >= orb_high:
                orb_extreme_first = 'high_first'
                break
            elif row['Low'] <= orb_low:
                orb_extreme_first = 'low_first'
                break
        
        if orb_extreme_first is None:
            continue
        
        # Which side breaks first after ORB
        post_orb_data = df.loc[orb_end:ny_cash_end]
        first_break = None
        high_break_time = None
        low_break_time = None
        midpoint_retest = False
        
        if not post_orb_data.empty:
            high_broken = False
            low_broken = False
            
            for idx, row in post_orb_data.iterrows():
                if not high_broken and row['High'] > orb_high:
                    high_broken = True
                    high_break_time = idx
                if not low_broken and row['Low'] < orb_low:
                    low_broken = True
                    low_break_time = idx
                
                if high_broken and low_broken:
                    break
            
            if high_broken and low_broken:
                first_break = 'high_breaks' if high_break_time < low_break_time else 'low_breaks'
            elif high_broken:
                first_break = 'high_breaks'
            elif low_broken:
                first_break = 'low_breaks'
            else:
                first_break = 'neither'
            
            # Check for midpoint retest after break
            if first_break == 'high_breaks':
                post_break_data = df.loc[high_break_time:ny_cash_end]
                for idx, row in post_break_data.iterrows():
                    if row['Low'] <= orb_midpoint:
                        midpoint_retest = True
                        break
            elif first_break == 'low_breaks':
                post_break_data = df.loc[low_break_time:ny_cash_end]
                for idx, row in post_break_data.iterrows():
                    if row['High'] >= orb_midpoint:
                        midpoint_retest = True
                        break
        else:
            first_break = 'neither'
        
        # Track combinations: ORB direction × Extreme first → Break direction, Close direction, Midpoint retest
        combo = orb_combos[orb_key][orb_direction][orb_extreme_first]
        combo['total'] += 1
        
        if first_break == 'high_breaks':
            combo['high_breaks'] += 1
        elif first_break == 'low_breaks':
            combo['low_breaks'] += 1
        else:
            combo['neither'] += 1
        
        if midpoint_retest:
            combo['midpoint_retest'] += 1
        
        if ny_cash_bullish:
            combo['bullish_close'] += 1
        else:
            combo['bearish_close'] += 1
        
        # Track combinations: ORB direction × Break direction → Midpoint retest, Close direction
        if first_break in ['high_breaks', 'low_breaks']:
            break_combo = orb_combos[orb_key][orb_direction][first_break]
            break_combo['total'] += 1
            
            if midpoint_retest:
                break_combo['midpoint_retest'] += 1
            
            if ny_cash_bullish:
                break_combo['bullish_close'] += 1
            else:
                break_combo['bearish_close'] += 1
        
        # Extension tracking
        if orb_high > 0:
            if ny_cash_high > orb_high:
                extension_pct = ((ny_cash_high - orb_high) / orb_high) * 100
                extension_stats[orb_key]['above_high']['extended'].append(extension_pct)
            else:
                extension_stats[orb_key]['above_high']['no_extension'] += 1
        
        if orb_low > 0:
            if ny_cash_low < orb_low:
                extension_pct = ((orb_low - ny_cash_low) / orb_low) * 100
                extension_stats[orb_key]['below_low']['extended'].append(extension_pct)
            else:
                extension_stats[orb_key]['below_low']['no_extension'] += 1
        
        # NEW: Track ORB → IB relationship
        orb_ib_tracking[orb_key].append({
            'orb_direction': orb_direction,
            'orb_extreme_first': orb_extreme_first,
            'orb_first_break': first_break,
            'ib_direction': ib_direction,
            'ib_extreme_first': ib_extreme_first,
            'ib_first_break': ib_first_break,
            'ib_close_position': ib_close_position
        })

# ============================================================
# PRINT RESULTS
# ============================================================

def print_combo_stats(combo, indent="  "):
    """Print combination statistics with percentages"""
    if combo['total'] == 0:
        print(f"{indent}No data")
        return
    
    total = combo['total']
    print(f"{indent}Total occurrences: {total}")
    
    # Break direction
    if 'high_breaks' in combo:
        high_pct = (combo['high_breaks'] / total) * 100
        low_pct = (combo['low_breaks'] / total) * 100
        neither_pct = (combo['neither'] / total) * 100
        print(f"{indent}  Break HIGH first : {combo['high_breaks']:4d} ({high_pct:5.1f}%)")
        print(f"{indent}  Break LOW first  : {combo['low_breaks']:4d} ({low_pct:5.1f}%)")
        print(f"{indent}  Neither breaks   : {combo['neither']:4d} ({neither_pct:5.1f}%)")
    
    # NY cash session close direction (9:30-4pm outcome)
    bull_pct = (combo['bullish_close'] / total) * 100
    bear_pct = (combo['bearish_close'] / total) * 100
    print(f"{indent}  → NY Session (9:30-4pm) closes BULLISH: {combo['bullish_close']:4d} ({bull_pct:5.1f}%)")
    print(f"{indent}  → NY Session (9:30-4pm) closes BEARISH: {combo['bearish_close']:4d} ({bear_pct:5.1f}%)")
    
    # Midpoint retest (if applicable)
    if 'midpoint_retest' in combo:
        retest_pct = (combo['midpoint_retest'] / total) * 100
        print(f"{indent}  Midpoint retest  : {combo['midpoint_retest']:4d} ({retest_pct:5.1f}%)")

def print_extension_stats(stats, label):
    """Print extension statistics showing % that extend vs don't"""
    extended = stats['extended']
    no_extension = stats['no_extension']
    total = len(extended) + no_extension
    
    if total == 0:
        print(f"  No data for {label}")
        return
    
    extension_pct = (len(extended) / total) * 100
    no_extension_pct = (no_extension / total) * 100
    
    print(f"\n  {label}:")
    print(f"    Total sessions   : {total}")
    print(f"    Sessions that extend      : {len(extended):4d} ({extension_pct:5.1f}%)")
    print(f"    Sessions with NO extension: {no_extension:4d} ({no_extension_pct:5.1f}%)")
    
    if extended:
        arr = np.array(extended)
        print(f"\n    For sessions that DO extend:")
        print(f"      Mean           : {arr.mean():.3f}%")
        print(f"      Median (50th)  : {np.percentile(arr, 50):.3f}%")
        print(f"      25th percentile: {np.percentile(arr, 25):.3f}%")
        print(f"      75th percentile: {np.percentile(arr, 75):.3f}%")
        print(f"      90th percentile: {np.percentile(arr, 90):.3f}%")
        print(f"      95th percentile: {np.percentile(arr, 95):.3f}%")
        print(f"      99th percentile: {np.percentile(arr, 99):.3f}%")

def print_orb_ib_predictions():
    """Print ORB → IB prediction statistics"""
    
    for orb_key in ['5min', '15min']:
        data = orb_ib_tracking[orb_key]
        
        print(f"\n\n{'=' * 80}")
        print(f"{orb_key.upper()} ORB → IB PREDICTIONS")
        print(f"{'=' * 80}")
        
        # 1. ORB direction → IB direction
        print(f"\n{'-' * 80}")
        print("Does ORB direction predict IB direction?")
        print(f"{'-' * 80}")
        
        for orb_dir in ['bullish', 'bearish']:
            matches = [d for d in data if d['orb_direction'] == orb_dir]
            if len(matches) == 0:
                continue
                
            ib_bull = sum(1 for d in matches if d['ib_direction'] == 'bullish')
            ib_bear = sum(1 for d in matches if d['ib_direction'] == 'bearish')
            
            print(f"\nWhen {orb_key} ORB is {orb_dir.upper()}: ({len(matches)} occurrences)")
            print(f"  → IB is BULLISH: {ib_bull:4d} ({100*ib_bull/len(matches):5.1f}%)")
            print(f"  → IB is BEARISH: {ib_bear:4d} ({100*ib_bear/len(matches):5.1f}%)")
        
        # 2. ORB extreme first → IB extreme first
        print(f"\n{'-' * 80}")
        print("Does ORB extreme pattern predict IB extreme pattern?")
        print(f"{'-' * 80}")
        
        for orb_ext in ['high_first', 'low_first']:
            matches = [d for d in data if d['orb_extreme_first'] == orb_ext]
            if len(matches) == 0:
                continue
                
            ib_high = sum(1 for d in matches if d['ib_extreme_first'] == 'high_first')
            ib_low = sum(1 for d in matches if d['ib_extreme_first'] == 'low_first')
            
            ext_label = "HIGH forms first" if orb_ext == 'high_first' else "LOW forms first"
            print(f"\nWhen {orb_key} ORB {ext_label}: ({len(matches)} occurrences)")
            print(f"  → IB HIGH forms first: {ib_high:4d} ({100*ib_high/len(matches):5.1f}%)")
            print(f"  → IB LOW forms first : {ib_low:4d} ({100*ib_low/len(matches):5.1f}%)")
        
        # 3. ORB break → IB break
        print(f"\n{'-' * 80}")
        print("Does ORB break direction predict IB break direction?")
        print(f"{'-' * 80}")
        
        for orb_break in ['high_breaks', 'low_breaks']:
            matches = [d for d in data if d['orb_first_break'] == orb_break]
            if len(matches) == 0:
                continue
                
            ib_high_break = sum(1 for d in matches if d['ib_first_break'] == 'high_breaks')
            ib_low_break = sum(1 for d in matches if d['ib_first_break'] == 'low_breaks')
            ib_neither = sum(1 for d in matches if d['ib_first_break'] == 'neither')
            
            break_label = "HIGH breaks first" if orb_break == 'high_breaks' else "LOW breaks first"
            print(f"\nWhen {orb_key} ORB {break_label}: ({len(matches)} occurrences)")
            print(f"  → IB HIGH breaks first: {ib_high_break:4d} ({100*ib_high_break/len(matches):5.1f}%)")
            print(f"  → IB LOW breaks first : {ib_low_break:4d} ({100*ib_low_break/len(matches):5.1f}%)")
            print(f"  → IB neither breaks   : {ib_neither:4d} ({100*ib_neither/len(matches):5.1f}%)")
        
        # 4. Combined: ORB direction + extreme → IB characteristics
        print(f"\n{'-' * 80}")
        print("Combined: ORB direction + extreme pattern → IB predictions")
        print(f"{'-' * 80}")
        
        for orb_dir in ['bullish', 'bearish']:
            for orb_ext in ['high_first', 'low_first']:
                matches = [d for d in data if d['orb_direction'] == orb_dir and d['orb_extreme_first'] == orb_ext]
                if len(matches) < 30:  # Skip if sample too small
                    continue
                
                ib_bull = sum(1 for d in matches if d['ib_direction'] == 'bullish')
                ib_high = sum(1 for d in matches if d['ib_extreme_first'] == 'high_first')
                
                ext_label = "HIGH first" if orb_ext == 'high_first' else "LOW first"
                print(f"\nWhen {orb_key} ORB is {orb_dir.upper()} + {ext_label}: ({len(matches)} occurrences)")
                print(f"  → IB is BULLISH       : {ib_bull:4d} ({100*ib_bull/len(matches):5.1f}%)")
                print(f"  → IB HIGH forms first : {ib_high:4d} ({100*ib_high/len(matches):5.1f}%)")

print("\n" + "=" * 80)
print("ENHANCED ORB & IB ANALYSIS: CONDITIONAL COMBINATIONS")
print("=" * 80)
print(f"\nData Period: {start_date.date()} to {df.index.max().date()}")

# ============================================================
# NEW SECTION 0: ORB → IB PREDICTIONS
# ============================================================

print_orb_ib_predictions()

# ============================================================
# SECTION 1: 5-MINUTE ORB COMBINATIONS
# ============================================================

print("\n\n" + "=" * 80)
print("SECTION 1: 5-MINUTE ORB CONDITIONAL COMBINATIONS")
print("=" * 80)

for direction in ['bullish', 'bearish']:
    direction_label = direction.upper()
    print(f"\n\n{'=' * 80}")
    print(f"When 5-min ORB is {direction_label}")
    print(f"{'=' * 80}")
    
    for extreme in ['high_first', 'low_first']:
        extreme_label = "HIGH forms first" if extreme == 'high_first' else "LOW forms first"
        print(f"\n{'-' * 80}")
        print(f"Condition: ORB {direction_label} + {extreme_label}")
        print(f"{'-' * 80}")
        print_combo_stats(orb_combos['5min'][direction][extreme], "  ")
    
    print(f"\n\n{'-' * 80}")
    print(f"When ORB is {direction_label}, breakdown by which side breaks first:")
    print(f"{'-' * 80}")
    
    for break_dir in ['high_breaks', 'low_breaks']:
        break_label = "HIGH breaks first" if break_dir == 'high_breaks' else "LOW breaks first"
        print(f"\n  Condition: ORB {direction_label} + {break_label}")
        print_combo_stats(orb_combos['5min'][direction][break_dir], "    ")

# ============================================================
# SECTION 2: 15-MINUTE ORB COMBINATIONS
# ============================================================

print("\n\n" + "=" * 80)
print("SECTION 2: 15-MINUTE ORB CONDITIONAL COMBINATIONS")
print("=" * 80)

for direction in ['bullish', 'bearish']:
    direction_label = direction.upper()
    print(f"\n\n{'=' * 80}")
    print(f"When 15-min ORB is {direction_label}")
    print(f"{'=' * 80}")
    
    for extreme in ['high_first', 'low_first']:
        extreme_label = "HIGH forms first" if extreme == 'high_first' else "LOW forms first"
        print(f"\n{'-' * 80}")
        print(f"Condition: ORB {direction_label} + {extreme_label}")
        print(f"{'-' * 80}")
        print_combo_stats(orb_combos['15min'][direction][extreme], "  ")
    
    print(f"\n\n{'-' * 80}")
    print(f"When ORB is {direction_label}, breakdown by which side breaks first:")
    print(f"{'-' * 80}")
    
    for break_dir in ['high_breaks', 'low_breaks']:
        break_label = "HIGH breaks first" if break_dir == 'high_breaks' else "LOW breaks first"
        print(f"\n  Condition: ORB {direction_label} + {break_label}")
        print_combo_stats(orb_combos['15min'][direction][break_dir], "    ")

# ============================================================
# SECTION 3: INITIAL BALANCE COMBINATIONS
# ============================================================

print("\n\n" + "=" * 80)
print("SECTION 3: INITIAL BALANCE CONDITIONAL COMBINATIONS")
print("=" * 80)

for direction in ['bullish', 'bearish']:
    direction_label = direction.upper()
    print(f"\n\n{'=' * 80}")
    print(f"When IB is {direction_label}")
    print(f"{'=' * 80}")
    
    # Extreme first × Close position
    for extreme in ['high_first', 'low_first']:
        extreme_label = "HIGH forms first" if extreme == 'high_first' else "LOW forms first"
        
        for close_pos in ['close_above_mid', 'close_below_mid']:
            close_label = "closes ABOVE midpoint" if close_pos == 'close_above_mid' else "closes BELOW midpoint"
            
            print(f"\n{'-' * 80}")
            print(f"Condition: IB {direction_label} + {extreme_label} + IB {close_label}")
            print(f"{'-' * 80}")
            print_combo_stats(ib_combos[direction][extreme][close_pos], "  ")
    
    # Just close position (simpler view)
    print(f"\n\n{'-' * 80}")
    print(f"When IB is {direction_label}, breakdown by IB close position:")
    print(f"{'-' * 80}")
    
    for close_pos in ['close_above_mid', 'close_below_mid']:
        close_label = "IB closes ABOVE midpoint" if close_pos == 'close_above_mid' else "IB closes BELOW midpoint"
        
        print(f"\n  Condition: IB {direction_label} + {close_label}")
        print_combo_stats(ib_combos[direction][close_pos], "    ")

# ============================================================
# SECTION 4: EXTENSION STATISTICS (ENHANCED)
# ============================================================

print("\n\n" + "=" * 80)
print("SECTION 4: RANGE EXTENSIONS (WITH NO-EXTENSION TRACKING)")
print("=" * 80)

print("\n" + "-" * 80)
print("5-MINUTE ORB EXTENSIONS")
print("-" * 80)
print_extension_stats(extension_stats['5min']['above_high'], "Extensions ABOVE 5-min ORB high")
print_extension_stats(extension_stats['5min']['below_low'], "Extensions BELOW 5-min ORB low")

print("\n\n" + "-" * 80)
print("15-MINUTE ORB EXTENSIONS")
print("-" * 80)
print_extension_stats(extension_stats['15min']['above_high'], "Extensions ABOVE 15-min ORB high")
print_extension_stats(extension_stats['15min']['below_low'], "Extensions BELOW 15-min ORB low")

print("\n\n" + "-" * 80)
print("INITIAL BALANCE EXTENSIONS")
print("-" * 80)
print_extension_stats(extension_stats['ib']['above_high'], "Extensions ABOVE IB high")
print_extension_stats(extension_stats['ib']['below_low'], "Extensions BELOW IB low")

# ============================================================
# SECTION 5: PREDICTIVE POWER SUMMARY
# ============================================================

print("\n\n" + "=" * 80)
print("SECTION 5: KEY PREDICTIVE PATTERNS FOR NY SESSION CLOSE (9:30-4PM)")
print("=" * 80)
print("\nHighest probability setups for predicting session close direction:")

print("\n" + "-" * 80)
print("5-MINUTE ORB STRONGEST SIGNALS")
print("-" * 80)

# Find strongest signals from 5min ORB
strongest_5min = []
for direction in ['bullish', 'bearish']:
    for extreme in ['high_first', 'low_first']:
        combo = orb_combos['5min'][direction][extreme]
        if combo['total'] >= 50:  # Minimum sample size
            bull_pct = (combo['bullish_close'] / combo['total']) * 100
            bear_pct = (combo['bearish_close'] / combo['total']) * 100
            strongest_5min.append({
                'condition': f"5min ORB {direction.upper()} + {extreme.replace('_', ' ').upper()}",
                'bull_pct': bull_pct,
                'bear_pct': bear_pct,
                'total': combo['total'],
                'dominant': 'BULLISH' if bull_pct > bear_pct else 'BEARISH',
                'edge': max(bull_pct, bear_pct)
            })

strongest_5min.sort(key=lambda x: x['edge'], reverse=True)

for i, signal in enumerate(strongest_5min[:5], 1):
    print(f"\n{i}. {signal['condition']}")
    print(f"   Sample size: {signal['total']}")
    print(f"   → Session closes {signal['dominant']}: {signal['edge']:.1f}%")
    print(f"   → Session closes {'BEARISH' if signal['dominant'] == 'BULLISH' else 'BULLISH'}: {100 - signal['edge']:.1f}%")

print("\n" + "-" * 80)
print("15-MINUTE ORB STRONGEST SIGNALS")
print("-" * 80)

# Find strongest signals from 15min ORB
strongest_15min = []
for direction in ['bullish', 'bearish']:
    for extreme in ['high_first', 'low_first']:
        combo = orb_combos['15min'][direction][extreme]
        if combo['total'] >= 50:
            bull_pct = (combo['bullish_close'] / combo['total']) * 100
            bear_pct = (combo['bearish_close'] / combo['total']) * 100
            strongest_15min.append({
                'condition': f"15min ORB {direction.upper()} + {extreme.replace('_', ' ').upper()}",
                'bull_pct': bull_pct,
                'bear_pct': bear_pct,
                'total': combo['total'],
                'dominant': 'BULLISH' if bull_pct > bear_pct else 'BEARISH',
                'edge': max(bull_pct, bear_pct)
            })

strongest_15min.sort(key=lambda x: x['edge'], reverse=True)

for i, signal in enumerate(strongest_15min[:5], 1):
    print(f"\n{i}. {signal['condition']}")
    print(f"   Sample size: {signal['total']}")
    print(f"   → Session closes {signal['dominant']}: {signal['edge']:.1f}%")
    print(f"   → Session closes {'BEARISH' if signal['dominant'] == 'BULLISH' else 'BULLISH'}: {100 - signal['edge']:.1f}%")

print("\n" + "-" * 80)
print("INITIAL BALANCE STRONGEST SIGNALS")
print("-" * 80)

# Find strongest signals from IB
strongest_ib = []
for direction in ['bullish', 'bearish']:
    for close_pos in ['close_above_mid', 'close_below_mid']:
        combo = ib_combos[direction][close_pos]
        if combo['total'] >= 50:
            bull_pct = (combo['bullish_close'] / combo['total']) * 100
            bear_pct = (combo['bearish_close'] / combo['total']) * 100
            close_label = "closes ABOVE midpoint" if close_pos == 'close_above_mid' else "closes BELOW midpoint"
            strongest_ib.append({
                'condition': f"IB {direction.upper()} + {close_label}",
                'bull_pct': bull_pct,
                'bear_pct': bear_pct,
                'total': combo['total'],
                'dominant': 'BULLISH' if bull_pct > bear_pct else 'BEARISH',
                'edge': max(bull_pct, bear_pct)
            })

strongest_ib.sort(key=lambda x: x['edge'], reverse=True)

for i, signal in enumerate(strongest_ib[:5], 1):
    print(f"\n{i}. {signal['condition']}")
    print(f"   Sample size: {signal['total']}")
    print(f"   → Session closes {signal['dominant']}: {signal['edge']:.1f}%")
    print(f"   → Session closes {'BEARISH' if signal['dominant'] == 'BULLISH' else 'BULLISH'}: {100 - signal['edge']:.1f}%")

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)